In [1]:
from bisect import bisect_left, bisect_right
from functools import cache

# 1) One-Dimensional DP

**In plain English.** Dynamic programming solves a big problem by solving smaller versions
of it once, writing the answers down, and building the big answer from them. It applies
when the same small questions come up again and again.

> **Analogy.** Climbing a staircase and counting the ways to reach each step: once you
> have written "step 5 can be reached in 8 ways" on the wall, nobody needs to recount it
> — step 6 and step 7 just read the number.

Write down four things before any code:

| Step | Question to answer | Example (House Robber) |
|---|---|---|
| **State** | what does `dp[i]` (or `dp[i][j]`) mean, in words? | best loot from houses `0..i` |
| **Transition** | how is a state built from smaller states? | `dp[i] = max(dp[i-1], dp[i-2] + nums[i])` |
| **Base case** | smallest states with known answers | `dp[0] = nums[0]` |
| **Answer** | which state(s) hold the final result? | `dp[n-1]` |

Two ways to implement the same recurrence:

- **Top-down (memoised recursion)** — write the recursion, add `@cache`. Easiest to get
  right; start here.
- **Bottom-up (tabulation)** — fill a table in dependency order. Lets you **compress
  space** when each row depends only on the previous one (`O(n²)` → `O(n)` memory).

| Family | Cue | Template state |
|---|---|---|
| 1D linear | "ways to reach / best up to index i" | `dp[i]` |
| Knapsack | pick items under a capacity / target sum | `dp[capacity]` (0/1: capacity **downwards**; unbounded: upwards) |
| Two sequences | compare / align two strings | `dp[i][j]` over prefixes |
| Grid | paths moving right/down | `dp[r][c]` |
| Intervals | split / merge a range, palindromes, burst | `dp[l][r]` by increasing length |
| State machine | holding / not holding / cooldown | `dp[i][state]` |

**Recognition signals**

- "number of ways", "minimum / maximum cost", "longest / shortest … subsequence"
- "can you reach / is it possible" with choices that overlap
- each decision depends only on a few earlier results (`i − 1`, `i − 2`, a smaller amount)
- greedy fails on a small counterexample
- brute force recursion recomputes the same arguments (draw the call tree!)

**Common bugs**

- A state that does not capture everything the future needs (add a dimension).
- Wrong base case (`dp[0] = 1` for "ways", `0` for "cost", `inf` for "minimum").
- Filling in the wrong order (a state used before it is computed).
- Off-by-one between "first i items" (length `n + 1`) and "index i" (length `n`).

**Don't confuse with**

| Pattern | Difference |
|---|---|
| Greedy | commits to the locally best choice; DP tries all choices and keeps the best |
| Backtracking | lists every solution; DP *counts* or *optimises* without listing |
| Divide and conquer | subproblems do not overlap (merge sort); DP's do |
| Prefix sums | a single running total, not a choice at each step |

**Learning path.** Fibonacci (the recurrence itself) → stairs with costs → take-or-skip
(House Robber) → circular and transformed variants → conditional transitions (Decode Ways)
→ DP from the back (Brainpower) → time-indexed DP (Tickets). Then "best ending at i"
(Kadane family), "can the prefix be built" (Word Break family), and LIS with its variants.

## 1.1 Fibonacci-style recurrences

`dp[i]` depends on a fixed number of earlier states (`i − 1`, `i − 2`). Write the
recurrence, then keep only the last few values in variables — O(1) space.

### 1.1.1 [70. Climbing Stairs](https://leetcode.com/problems/climbing-stairs/) — ★★★ · Easy

**Problem.** Climb `n` steps taking 1 or 2 at a time; how many distinct ways?  
**Signal.** Ways to reach step `i` = ways to reach `i−1` + ways to reach `i−2` → Fibonacci.  
*New here: The basic recurrence, solved with two rolling variables.*

**Fine print.** Ask before coding:
- Does order matter (1+2 vs 2+1)? → Yes, they are different ways.
- `n ≥ 1`? → Yes, up to 45, so the answer fits in 32 bits.

> 💡 **Key insight.** Your last move onto step `i` came from step `i − 1` or step `i − 2`, so `ways(i) = ways(i − 1) + ways(i − 2)`.

**Brute force →** Plain recursion → O(2ⁿ) because it recomputes the same steps. Memoisation → O(n) time and space. Two variables → O(n) time, O(1) space.

**Steps**
1. Start with ways(0) = 1 and ways(1) = 1.
2. Roll forward: `prev, cur = cur, prev + cur`.
3. After n − 1 rolls, `cur` is the answer.

**Complexity**

- **Time:** O(n).
- **Space:** O(1).

In [2]:
def climb_stairs(n: int) -> int:
    """Number of 1/2-step ways to climb n stairs."""
    # ways to reach step 0 and step 1
    prev, cur = 1, 1
    for _ in range(n - 1):
        # arrive from one step below or two steps below
        prev, cur = cur, prev + cur
    return cur


assert climb_stairs(2) == 2
assert climb_stairs(3) == 3
assert climb_stairs(45) == 1836311903
print(climb_stairs(5))

8


### 1.1.2 [509. Fibonacci Number](https://leetcode.com/problems/fibonacci-number/) — ★ · Easy

**Problem.** Compute `F(n)` with `F(0) = 0`, `F(1) = 1`.  
**Signal.** The canonical overlapping-subproblem recursion.  
*New here: Seeing *why* plain recursion explodes (overlapping subproblems).*

**Fine print.** Ask before coding:
- `n ≤ 30`? → Yes, so even recursion is fine, but O(n) with two variables is the target.

> 💡 **Key insight.** `F(n − 1)` already computes `F(n − 2)`, so naive recursion repeats work exponentially; each value only needs computing once.

**Brute force →** `fib(n − 1) + fib(n − 2)` recursion → O(φⁿ). Iteration → O(n).

**Steps**
1. `a, b = 0, 1` (F(0), F(1)).
2. Repeat n times: `a, b = b, a + b`.
3. Return `a`.

**Complexity**

- **Time:** O(n).
- **Space:** O(1).

In [3]:
def fib(n: int) -> int:
    """n-th Fibonacci number."""
    # a = F(i), b = F(i + 1)
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a


assert [fib(n) for n in range(8)] == [0, 1, 1, 2, 3, 5, 8, 13]
print(fib(30))

832040


### 1.1.3 [1137. N-th Tribonacci Number](https://leetcode.com/problems/n-th-tribonacci-number/) — ★ · Easy

**Problem.** `T(n) = T(n−1) + T(n−2) + T(n−3)` with `T(0)=0, T(1)=1, T(2)=1`.  
**Signal.** Fibonacci with three terms.  
*New here: Rolling a window of three values.*

**Fine print.** Ask before coding:
- `n ≤ 37`? → Yes, so the answer fits in 32 bits.

> 💡 **Key insight.** The state needs exactly the last three values.

**Brute force →** Recursion → O(3ⁿ). Rolling variables → O(n).

**Steps**
1. `a, b, c = 0, 1, 1`.
2. Roll: `a, b, c = b, c, a + b + c`, n times.
3. Return `a`.

**Complexity**

- **Time:** O(n).
- **Space:** O(1).

In [4]:
def tribonacci(n: int) -> int:
    """n-th Tribonacci number."""
    a, b, c = 0, 1, 1
    for _ in range(n):
        # shift the three-value window forward
        a, b, c = b, c, a + b + c
    return a


assert tribonacci(4) == 4
assert tribonacci(25) == 1389537
print(tribonacci(25))

1389537


### 1.1.4 [746. Min Cost Climbing Stairs](https://leetcode.com/problems/min-cost-climbing-stairs/) — ★★ · Easy

**Problem.** Pay `cost[i]` to step on stair `i`, then climb 1 or 2; start at index 0 or 1. Minimum cost to get past the top?  
**Signal.** `dp[i]` = cheapest cost to stand on step `i`.  
*New here: From counting to *minimising*: `min` replaces `+`.*

**Fine print.** Ask before coding:
- Is "the top" one past the last index? → Yes.
- Can you start at index 0 or 1 for free? → Yes.

> 💡 **Key insight.** The cheapest way to stand on step `i` is its own cost plus the cheaper of the two steps you could have come from.

**Brute force →** Try all step sequences → O(2ⁿ). DP → O(n).

**Steps**
1. `dp[0] = cost[0]`, `dp[1] = cost[1]`.
2. `dp[i] = cost[i] + min(dp[i − 1], dp[i − 2])`.
3. The top is reached from either of the last two steps: `min(dp[n − 1], dp[n − 2])`.

**Complexity**

- **Time:** O(n).
- **Space:** O(1) with two variables.

In [5]:
def min_cost_climbing_stairs(cost: list[int]) -> int:
    """Cheapest way past the last stair."""
    # cheapest cost to stand on the previous two steps
    a, b = cost[0], cost[1]
    for c in cost[2:]:
        a, b = b, c + min(a, b)
    # jump to the top from either of the last two steps
    return min(a, b)


assert min_cost_climbing_stairs([10, 15, 20]) == 15
assert min_cost_climbing_stairs([1, 100, 1, 1, 1, 100, 1, 1, 100, 1]) == 6
print(min_cost_climbing_stairs([10, 15, 20]))

15


### 1.1.5 [198. House Robber](https://leetcode.com/problems/house-robber/) — ★★★ · Medium

**Problem.** Rob houses in a row without robbing two adjacent ones; maximise loot.  
**Signal.** Take-or-skip at each index → `dp[i] = max(dp[i−1], dp[i−2] + nums[i])`.  
*New here: The take-or-skip recurrence.*

**Fine print.** Ask before coding:
- Non-negative values? → Yes.
- Can the array be empty? → `n ≥ 1` on LeetCode.

> 💡 **Key insight.** At each house you either skip it (keep the best so far) or rob it (best from two houses back plus its value).

**Brute force →** Try every non-adjacent subset → O(2ⁿ). DP → O(n).

**Steps**
1. `prev` = best up to `i − 2`, `cur` = best up to `i − 1`.
2. For each house: new best = `max(cur, prev + x)`.
3. Shift `prev, cur`.
4. Return `cur`.

**Complexity**

- **Time:** O(n).
- **Space:** O(1).

In [6]:
def rob(nums: list[int]) -> int:
    """Max loot from non-adjacent houses."""
    # best up to two houses back, best up to the previous house
    prev, cur = 0, 0
    for x in nums:
        # skip this house, or rob it on top of prev
        prev, cur = cur, max(cur, prev + x)
    return cur


assert rob([1, 2, 3, 1]) == 4
assert rob([2, 7, 9, 3, 1]) == 12
print(rob([2, 7, 9, 3, 1]))

12


### 1.1.6 [213. House Robber II](https://leetcode.com/problems/house-robber-ii/) — ★★★ · Medium

**Problem.** Same as House Robber, but houses are in a circle (first and last are adjacent).  
**Signal.** Break the circle: the answer excludes either the first or the last house.  
*New here: Breaking a cycle into two linear cases.*

**Fine print.** Ask before coding:
- Are the first and last houses adjacent? → Yes, so solve two linear cases.
- Only one house? → Rob it.

> 💡 **Key insight.** The first and last houses can't both be robbed, so the answer is the better of "ignore the first house" and "ignore the last house" — both linear problems.

**Brute force →** DP with an extra "was the first robbed" flag → same O(n), more state. Two linear runs is simpler.

**Steps**
1. A single house → its value.
2. Run House Robber on `nums[1:]`.
3. Run House Robber on `nums[:-1]`.
4. Return the larger.

**Complexity**

- **Time:** O(n).
- **Space:** O(n) for the slices (O(1) with index ranges).

In [7]:
def rob_circle(nums: list[int]) -> int:
    """Max loot when houses form a circle."""
    if len(nums) == 1:
        return nums[0]
    # first and last can't both be robbed: drop one of them
    return max(rob(nums[1:]), rob(nums[:-1]))


assert rob_circle([2, 3, 2]) == 3
assert rob_circle([1, 2, 3, 1]) == 4
assert rob_circle([1, 2, 3]) == 3
print(rob_circle([2, 3, 2]))

3


### 1.1.7 [740. Delete and Earn](https://leetcode.com/problems/delete-and-earn/) — ★ · Medium

**Problem.** Taking value `x` earns `x` but deletes every `x−1` and `x+1`. Maximise points.  
**Signal.** Bucket by value → House Robber over the value axis.  
*New here: Transforming the input so a known DP applies.*

**Fine print.** Ask before coding:
- Does taking `x` delete only neighbouring values, not other copies of `x`? → Yes, so take every copy of `x`.
- Values up to 10⁴? → Yes, so a value-indexed array works.

> 💡 **Key insight.** Taking all copies of `v` earns `v × count(v)` and forbids `v − 1` and `v + 1` — exactly House Robber on an array indexed by value.

**Brute force →** Try every subset of values → O(2^m). Bucket + House Robber → O(n + max).

**Steps**
1. `points[v] = v × count(v)`.
2. Run House Robber on `points`.

**Complexity**

- **Time:** O(n + max value).
- **Space:** O(max value).

In [8]:
def delete_and_earn(nums: list[int]) -> int:
    """Max points under the delete-neighbours rule."""
    # total earned by taking every copy of each value
    points = [0] * (max(nums) + 1)
    for x in nums:
        points[x] += x
    # neighbouring values behave like neighbouring houses
    return rob(points)


assert delete_and_earn([3, 4, 2]) == 6
assert delete_and_earn([2, 2, 3, 3, 3, 4]) == 9
print(delete_and_earn([2, 2, 3, 3, 3, 4]))

9


### 1.1.8 [91. Decode Ways](https://leetcode.com/problems/decode-ways/) — ★★★ · Medium

**Problem.** Count decodings of a digit string where `A=1 … Z=26`.  
**Signal.** Last step consumes one digit (`1–9`) or two digits (`10–26`) → Fibonacci with conditions.  
*New here: Transitions that only apply under conditions.*

**Fine print.** Ask before coding:
- Leading zeros, or a `0` that cannot be paired? → 0 decodings.
- Does the answer fit in 32 bits? → Yes, guaranteed.

> 💡 **Key insight.** The last letter of a decoding uses either the last digit (if it is not 0) or the last two digits (if they form 10–26), so the counts add like Fibonacci — but only when allowed.

**Brute force →** Recursion over both choices → O(2ⁿ). DP → O(n).

**Steps**
1. `prev` = ways for the prefix two shorter, `cur` = ways for the prefix one shorter (0 if `s[0] == "0"`).
2. New count = (`cur` if this digit isn't 0) + (`prev` if the last two digits are 10–26).
3. Shift and repeat.

**Complexity**

- **Time:** O(n).
- **Space:** O(1).

⚠️ **Trap.** A `0` can only be decoded as part of `10` or `20`.

In [9]:
def num_decodings(s: str) -> int:
    """Number of ways to decode s."""
    # ways for the empty prefix and for the first character
    prev, cur = 1, 1 if s[0] != "0" else 0
    for i in range(1, len(s)):
        nxt = 0
        # last letter uses one digit
        if s[i] != "0":
            nxt += cur
        # last letter uses two digits
        if 10 <= int(s[i - 1 : i + 1]) <= 26:
            nxt += prev
        prev, cur = cur, nxt
    return cur


assert num_decodings("12") == 2
assert num_decodings("226") == 3
assert num_decodings("06") == 0
assert num_decodings("2101") == 1
print(num_decodings("226"))

3


### 1.1.9 [2140. Solving Questions With Brainpower](https://leetcode.com/problems/solving-questions-with-brainpower/) — ★ · Medium

**Problem.** Question `i` gives `points` but makes you skip the next `brainpower` questions. Max points?  
**Signal.** Decision at each index with a jump → DP from the back.  
*New here: Filling the table right to left because choices jump forward.*

**Fine print.** Ask before coding:
- Can you skip a question? → Yes, without penalty.
- Can sums exceed 32 bits? → Yes, up to about 10¹⁰.

> 💡 **Key insight.** Solving question `i` sends you to `i + brainpower + 1`, so if you know the best score from every later index, `i` is a two-way choice.

**Brute force →** Recursion over solve/skip → O(2ⁿ). Backward DP → O(n).

**Steps**
1. `dp[n] = 0`.
2. For `i` from n − 1 down to 0: `dp[i] = max(dp[i + 1], points + dp[min(n, i + skip + 1)])`.
3. Return `dp[0]`.

**Complexity**

- **Time:** O(n).
- **Space:** O(n).

In [10]:
def most_points(questions: list[list[int]]) -> int:
    """Best total score with skip penalties."""
    n = len(questions)
    # dp[i] = best score using questions i..n-1
    dp = [0] * (n + 1)
    for i in range(n - 1, -1, -1):
        points, skip = questions[i]
        # skip it, or solve it and jump over the next `skip` questions
        dp[i] = max(dp[i + 1], points + dp[min(n, i + skip + 1)])
    return dp[0]


assert most_points([[3, 2], [4, 3], [4, 4], [2, 5]]) == 5
assert most_points([[1, 1], [2, 2], [3, 3], [4, 4], [5, 5]]) == 7
print(most_points([[3, 2], [4, 3], [4, 4], [2, 5]]))

5


### 1.1.10 [983. Minimum Cost For Tickets](https://leetcode.com/problems/minimum-cost-for-tickets/) — ★ · Medium

**Problem.** Travel days in a year; 1-, 7- and 30-day passes with given costs. Minimum total cost?  
**Signal.** `dp[day]` = cheapest cost to cover all travel days up to `day`.  
*New here: Indexing the DP by *time* instead of by array position.*

**Fine print.** Ask before coding:
- Strictly increasing days in `1..365`? → Yes.
- Can a pass cover days you do not travel? → Yes, and that is allowed.

> 💡 **Key insight.** If today is a travel day, some pass must cover it; the cheapest choice is the best of "a pass ending today" plus the cost before that pass started.

**Brute force →** Try all pass combinations → exponential. DP over days → O(365 · 3).

**Steps**
1. `dp[0] = 0`.
2. Non-travel day: `dp[d] = dp[d − 1]`.
3. Travel day: `dp[d] = min(cost + dp[max(0, d − span)])` over the three passes.
4. Return `dp[last day]`.

**Complexity**

- **Time:** O(last day).
- **Space:** O(last day).

In [11]:
def mincost_tickets(days: list[int], costs: list[int]) -> int:
    """Cheapest pass combination covering all days."""
    travel = set(days)
    # dp[d] = cheapest cover of all travel days up to day d
    dp = [0] * (days[-1] + 1)
    for d in range(1, days[-1] + 1):
        # no travel: nothing new to pay for
        if d not in travel:
            dp[d] = dp[d - 1]
            continue
        # a pass of `span` days ending today
        dp[d] = min(dp[max(0, d - span)] + c for span, c in zip((1, 7, 30), costs))
    return dp[-1]


assert mincost_tickets([1, 4, 6, 7, 8, 20], [2, 7, 15]) == 11
assert mincost_tickets([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 30, 31], [2, 7, 15]) == 17
print(mincost_tickets([1, 4, 6, 7, 8, 20], [2, 7, 15]))

11


## 1.2 Kadane and subarray DP

The state is "best subarray **ending at** `i`". Each element either extends the previous
best or starts fresh; the global answer is the best over all `i`.

### 1.2.1 [53. Maximum Subarray](https://leetcode.com/problems/maximum-subarray/) — ★★★ · Medium

**Problem.** Largest sum of a non-empty contiguous subarray.  
**Signal.** Best subarray *ending at i* → Kadane's algorithm.  
*New here: The "ending at i" state (Kadane).*

**Fine print.** Ask before coding:
- Non-empty subarray? → Yes, so an all-negative array returns its largest element.
- Return the sum, or also the indices? → The sum; ask about the indices as a follow-up.

> 💡 **Key insight.** A subarray ending at `i` either extends the best one ending at `i − 1` or starts fresh at `i` — whichever is larger.

**Brute force →** All subarrays → O(n²). Kadane → O(n).

**Steps**
1. `cur` = best sum ending here; `best` = best overall.
2. For each `x`: `cur = max(x, cur + x)`.
3. `best = max(best, cur)`.

**Complexity**

- **Time:** O(n).
- **Space:** O(1).

⚠️ **Trap.** Start from `nums[0]`, not 0 — an all-negative array must return its largest element.

In [12]:
def max_sub_array(nums: list[int]) -> int:
    """Largest contiguous subarray sum."""
    best = cur = nums[0]
    for x in nums[1:]:
        # extend the previous subarray, or start a new one here
        cur = max(x, cur + x)
        best = max(best, cur)
    return best


assert max_sub_array([-2, 1, -3, 4, -1, 2, 1, -5, 4]) == 6
assert max_sub_array([1]) == 1
assert max_sub_array([-3, -1, -2]) == -1
print(max_sub_array([-2, 1, -3, 4, -1, 2, 1, -5, 4]))

6


### 1.2.2 [152. Maximum Product Subarray](https://leetcode.com/problems/maximum-product-subarray/) — ★★★ · Medium

**Problem.** Largest product of a contiguous subarray.  
**Signal.** Kadane for products — a negative number swaps max and min.  
*New here: Keeping two states (max and min) because signs can flip.*

**Fine print.** Ask before coding:
- Zeros and negatives? → Yes, so track both the max and the min.
- Does the answer fit in 32 bits? → Yes, guaranteed.

> 💡 **Key insight.** Multiplying by a negative turns the most negative product into the largest, so you must also remember the minimum product ending here.

**Brute force →** All subarrays → O(n²). Max/min Kadane → O(n).

**Steps**
1. `hi`, `lo` = max and min product ending here.
2. For each `x`, candidates are `x`, `hi·x`, `lo·x`.
3. New `hi` = max of them, new `lo` = min of them.
4. Track the best `hi`.

**Complexity**

- **Time:** O(n).
- **Space:** O(1).

In [13]:
def max_product(nums: list[int]) -> int:
    """Largest contiguous subarray product."""
    best = hi = lo = nums[0]
    for x in nums[1:]:
        # start fresh, or extend the largest / smallest product
        candidates = (x, hi * x, lo * x)
        hi, lo = max(candidates), min(candidates)
        best = max(best, hi)
    return best


assert max_product([2, 3, -2, 4]) == 6
assert max_product([-2, 0, -1]) == 0
assert max_product([-2, 3, -4]) == 24
print(max_product([2, 3, -2, 4]))

6


### 1.2.3 [918. Maximum Sum Circular Subarray](https://leetcode.com/problems/maximum-sum-circular-subarray/) — ★★ · Medium

**Problem.** Maximum subarray sum where the array is circular.  
**Signal.** A wrapping subarray = total − (minimum subarray in the middle).  
*New here: Complement reasoning: max wrapping = total − min middle.*

**Fine print.** Ask before coding:
- Non-empty? → Yes.
- Can an element be used only once (no full wrap twice)? → Yes.

> 💡 **Key insight.** A subarray that wraps around leaves out a contiguous middle block, so its sum is `total − (that block)`, maximised when the block is the minimum subarray.

**Brute force →** Every start and length on the doubled array → O(n²). Two Kadanes → O(n).

**Steps**
1. Kadane for the maximum subarray.
2. Kadane for the minimum subarray.
3. If all numbers are negative (`best_max < 0`), return `best_max`.
4. Otherwise return `max(best_max, total − best_min)`.

**Complexity**

- **Time:** O(n).
- **Space:** O(1).

⚠️ **Trap.** With all negatives, `total − best_min` corresponds to an empty subarray (0), which is not allowed.

In [14]:
def max_subarray_sum_circular(nums: list[int]) -> int:
    """Largest circular subarray sum."""
    cur_max = cur_min = 0
    best_max, best_min = nums[0], nums[0]
    for x in nums:
        # Kadane for the maximum and the minimum at once
        cur_max = max(x, cur_max + x)
        cur_min = min(x, cur_min + x)
        best_max = max(best_max, cur_max)
        best_min = min(best_min, cur_min)
    # wrapping answer = everything except the worst middle block
    return best_max if best_max < 0 else max(best_max, sum(nums) - best_min)


assert max_subarray_sum_circular([1, -2, 3, -2]) == 3
assert max_subarray_sum_circular([5, -3, 5]) == 10
assert max_subarray_sum_circular([-3, -2, -3]) == -2
print(max_subarray_sum_circular([5, -3, 5]))

10


### 1.2.4 [413. Arithmetic Slices](https://leetcode.com/problems/arithmetic-slices/) — ★ · Medium

**Problem.** Count contiguous subarrays of length ≥ 3 with constant difference.  
**Signal.** `dp[i]` = number of arithmetic slices *ending* at `i`.  
*New here: Counting with an "ending at i" state.*

**Fine print.** Ask before coding:
- At least 3 elements? → Yes.
- Contiguous? → Yes.

> 💡 **Key insight.** If the difference continues at `i`, every slice ending at `i − 1` extends by one, plus one new slice of length 3: `dp[i] = dp[i − 1] + 1`.

**Brute force →** Check every subarray → O(n²). DP → O(n).

**Steps**
1. `cur` = slices ending at the current index.
2. If the last three numbers keep the difference, `cur += 1`; else `cur = 0`.
3. Add `cur` to the total.

**Complexity**

- **Time:** O(n).
- **Space:** O(1).

In [15]:
def number_of_arithmetic_slices(nums: list[int]) -> int:
    """Count arithmetic subarrays of length >= 3."""
    total = cur = 0
    for i in range(2, len(nums)):
        # same difference: every slice ending at i-1 extends, plus one new
        if nums[i] - nums[i - 1] == nums[i - 1] - nums[i - 2]:
            cur += 1
            total += cur
        else:
            cur = 0
    return total


assert number_of_arithmetic_slices([1, 2, 3, 4]) == 3
assert number_of_arithmetic_slices([1]) == 0
print(number_of_arithmetic_slices([1, 2, 3, 4, 6, 8, 10]))

6


## 1.3 Word and partition DP

The state is "can / how many ways / best way to build the **prefix** of length `i`". The
transition tries every last piece (a word, a square, a part) that ends at `i`.

### 1.3.1 [139. Word Break](https://leetcode.com/problems/word-break/) — ★★★ · Medium

**Problem.** Can `s` be segmented into dictionary words?  
**Signal.** `dp[i]` = prefix of length `i` is segmentable.  
*New here: A boolean prefix DP trying every last piece.*

**Fine print.** Ask before coding:
- Can dictionary words be reused? → Yes.
- Unique dictionary words? → Yes.

> 💡 **Key insight.** The prefix `s[:i]` is segmentable iff some shorter segmentable prefix `s[:j]` is followed by a dictionary word `s[j:i]`.

**Brute force →** Recursion trying every split → O(2ⁿ). DP → O(n · L) checks.

**Steps**
1. `dp[0] = True` (empty prefix).
2. For each `i`, try `j` from `i − longest` to `i − 1`.
3. If `dp[j]` and `s[j:i]` is a word, set `dp[i] = True` and stop.
4. Return `dp[n]`.

**Complexity**

- **Time:** O(n · L) checks, each O(L) to slice → O(n · L²).
- **Space:** O(n).

In [16]:
def word_break(s: str, word_dict: list[str]) -> bool:
    """True if s splits into dictionary words."""
    words = set(word_dict)
    longest = max(map(len, words))
    # dp[i]: the first i characters can be segmented
    dp = [True] + [False] * len(s)
    for i in range(1, len(s) + 1):
        # the last word s[j:i] is at most `longest` long
        for j in range(max(0, i - longest), i):
            if dp[j] and s[j:i] in words:
                dp[i] = True
                break
    return dp[-1]


assert word_break("leetcode", ["leet", "code"])
assert word_break("applepenapple", ["apple", "pen"])
assert not word_break("catsandog", ["cats", "dog", "sand", "and", "cat"])
print(word_break("leetcode", ["leet", "code"]))

True


### 1.3.2 [140. Word Break II](https://leetcode.com/problems/word-break-ii/) — ★ · Hard

**Problem.** Return every sentence formed by segmenting `s` into dictionary words.  
**Signal.** Word Break with memoised lists of suffix sentences.  
*New here: Top-down memoisation that returns *lists* instead of numbers.*

**Fine print.** Ask before coding:
- Any order? → Yes.
- How large can the output be? → Up to 10⁵ sentences, so memoise on the suffix.

> 💡 **Key insight.** All sentences for the suffix starting at `i` are "a word matching at `i`" + "any sentence for the rest", and the same suffixes recur, so cache them.

**Brute force →** Backtracking without memo → recomputes shared suffixes. Memoised recursion computes each suffix once (the output itself can still be exponential).

**Steps**
1. `sentences(i)` returns all sentences for `s[i:]`; `sentences(n) = [""]`.
2. For each `j`, if `s[i:j]` is a word, prefix it to every sentence of `sentences(j)`.
3. `@cache` stores each suffix's list.
4. Return `sentences(0)`.

**Complexity**

- **Time:** Exponential in the worst case (output size).
- **Space:** Exponential in the worst case (output size).

In [17]:
def word_break_ii(s: str, word_dict: list[str]) -> list[str]:
    """All segmentations of s into words."""
    words = set(word_dict)

    @cache
    def sentences(i: int) -> list[str]:
        """All segmentations of s[i:]."""
        if i == len(s):
            return [""]
        out = []
        for j in range(i + 1, len(s) + 1):
            # a word here + any segmentation of the rest
            if s[i:j] in words:
                for rest in sentences(j):
                    out.append(s[i:j] + (" " + rest if rest else ""))
        return out

    return sentences(0)


res = word_break_ii("catsanddog", ["cat", "cats", "and", "sand", "dog"])
assert sorted(res) == ["cat sand dog", "cats and dog"]
assert word_break_ii("catsandog", ["cats", "dog", "sand", "and", "cat"]) == []
print(res)

['cat sand dog', 'cats and dog']


### 1.3.3 [279. Perfect Squares](https://leetcode.com/problems/perfect-squares/) — ★★ · Medium

**Problem.** Fewest perfect squares summing to `n`.  
**Signal.** Unbounded knapsack / coin change with coins `1, 4, 9, …`.  
*New here: Minimising a count over "last piece" choices.*

**Fine print.** Ask before coding:
- Is 1 a perfect square? → Yes, so an answer always exists.
- `n ≤ 10⁴`? → Yes.

> 💡 **Key insight.** The last square used is some `s ≤ i`, so `dp[i] = 1 + min(dp[i − s])`.

**Brute force →** Recursion over every square → exponential. DP → O(n √n). (BFS on remainders also works.)

**Steps**
1. `dp[0] = 0`, everything else large.
2. For each `i`, try every square `s ≤ i`: `dp[i] = min(dp[i], dp[i − s] + 1)`.
3. Return `dp[n]`.

**Complexity**

- **Time:** O(n √n).
- **Space:** O(n).

In [18]:
def num_squares(n: int) -> int:
    """Fewest squares summing to n."""
    squares = [k * k for k in range(1, int(n**0.5) + 1)]
    # dp[i] = fewest squares summing to i (n is a safe upper bound)
    dp = [0] + [n] * n
    for i in range(1, n + 1):
        # the last square used is s
        for s in squares:
            if s > i:
                break
            dp[i] = min(dp[i], dp[i - s] + 1)
    return dp[n]


assert num_squares(12) == 3
assert num_squares(13) == 2
print(num_squares(12))

3


### 1.3.4 [343. Integer Break](https://leetcode.com/problems/integer-break/) — ★ · Medium

**Problem.** Split `n ≥ 2` into at least two positive integers maximising their product.  
**Signal.** `dp[i] = max over j of max(j, dp[j]) × max(i−j, dp[i−j])`.  
*New here: Each part may be "kept whole" or "split further".*

**Fine print.** Ask before coding:
- At least two parts? → Yes, which is why `n = 2` gives 1 and `n = 3` gives 2.

> 💡 **Key insight.** For each first cut `j`, both pieces can either stay whole or be split again, so take `max(j, dp[j])` for each side.

**Brute force →** Enumerate all partitions → exponential. DP → O(n²). (Math: use as many 3s as possible.)

**Steps**
1. `dp[1] = 1`.
2. For each `i`, try every cut `j` up to `i / 2`.
3. `dp[i] = max(dp[i], max(j, dp[j]) · max(i − j, dp[i − j]))`.
4. Return `dp[n]`.

**Complexity**

- **Time:** O(n²).
- **Space:** O(n).

In [19]:
def integer_break(n: int) -> int:
    """Max product of a split of n into >= 2 parts."""
    dp = [0, 1] + [0] * (n - 1)
    for i in range(2, n + 1):
        for j in range(1, i // 2 + 1):
            # each side either stays whole or is split further
            dp[i] = max(dp[i], max(j, dp[j]) * max(i - j, dp[i - j]))
    return dp[n]


assert integer_break(2) == 1
assert integer_break(10) == 36
print(integer_break(10))

36


### 1.3.5 [96. Unique Binary Search Trees](https://leetcode.com/problems/unique-binary-search-trees/) — ★ · Medium

**Problem.** Number of structurally unique BSTs with keys `1..n`.  
**Signal.** Choose the root, multiply the counts of left and right subtrees → Catalan numbers.  
*New here: Multiplying independent sub-counts (product rule).*

**Fine print.** Ask before coding:
- Count structures, not values? → Yes.
- `n ≤ 19`? → Yes, so the Catalan number fits in 32 bits.

> 💡 **Key insight.** With root `r`, the left subtree is any BST of `r − 1` keys and the right any BST of `n − r` keys, chosen independently, so their counts multiply.

**Brute force →** Build all trees → Catalan-many. DP on counts → O(n²).

**Steps**
1. `g[0] = 1` (empty tree).
2. For each size `nodes`, sum over roots: `g[root − 1] · g[nodes − root]`.
3. Return `g[n]`.

**Complexity**

- **Time:** O(n²).
- **Space:** O(n).

In [20]:
def num_trees(n: int) -> int:
    """Count of unique BSTs with n keys."""
    # g[k] = number of BSTs with k keys
    g = [1] + [0] * n
    for nodes in range(1, n + 1):
        for root in range(1, nodes + 1):
            # left and right subtrees are chosen independently
            g[nodes] += g[root - 1] * g[nodes - root]
    return g[n]


assert num_trees(3) == 5
assert num_trees(1) == 1
print([num_trees(n) for n in range(1, 8)])

[1, 2, 5, 14, 42, 132, 429]


## 1.4 Longest increasing subsequence

LIS-type state: `dp[i]` = best subsequence **ending at** `i`, built from any earlier `j`
that may precede it. O(n²) is the default; patience sorting with binary search gives
O(n log n) for the plain length.

### 1.4.1 [300. Longest Increasing Subsequence](https://leetcode.com/problems/longest-increasing-subsequence/) — ★★★ · Medium

**Problem.** Length of the longest strictly increasing subsequence.  
**Signal.** O(n²) DP or O(n log n) patience sorting.  
*New here: Patience sorting: a sorted `tails` array + binary search.*

**Fine print.** Ask before coding:
- Strictly increasing? → Yes.
- Subsequence, not contiguous? → Yes.
- O(n log n)? → That is the follow-up.

> 💡 **Key insight.** For each length, only the *smallest possible tail* matters (it is easiest to extend), and those tails stay sorted, so each new number updates one tail found by binary search.

**Brute force →** All subsequences → O(2ⁿ). `dp[i] = 1 + max(dp[j])` over smaller earlier values → O(n²). Tails + binary search → O(n log n).

**Steps**
1. `tails[k]` = smallest tail of an increasing subsequence of length `k + 1`.
2. For each `x`, find the first tail ≥ `x` with `bisect_left`.
3. Replace it with `x`, or append if `x` is larger than all tails.
4. The answer is `len(tails)`.

**Complexity**

- **Time:** O(n log n).
- **Space:** O(n).

⚠️ **Trap.** `tails` is not an actual subsequence — only its length is meaningful.

In [21]:
def length_of_lis(nums: list[int]) -> int:
    """Length of the longest strictly increasing subsequence."""
    # tails[k] = smallest tail of an increasing subsequence of length k + 1
    tails: list[int] = []
    for x in nums:
        # first tail >= x (strictly increasing, so bisect_left)
        i = bisect_left(tails, x)
        # x extends the longest subsequence
        if i == len(tails):
            tails.append(x)
        # x is a smaller tail for length i + 1
        else:
            tails[i] = x
    return len(tails)


assert length_of_lis([10, 9, 2, 5, 3, 7, 101, 18]) == 4
assert length_of_lis([0, 1, 0, 3, 2, 3]) == 4
assert length_of_lis([7, 7, 7, 7]) == 1
print(length_of_lis([10, 9, 2, 5, 3, 7, 101, 18]))

4


### 1.4.2 [673. Number of Longest Increasing Subsequence](https://leetcode.com/problems/number-of-longest-increasing-subsequence/) — ★ · Medium

**Problem.** How many longest increasing subsequences are there?  
**Signal.** O(n²) LIS that also counts ways.  
*New here: Carrying a count alongside the best length.*

**Fine print.** Ask before coding:
- Strictly increasing? → Yes.
- Does the answer fit in 32 bits? → Yes.

> 💡 **Key insight.** For each `i`, the number of LIS ending there is the sum of the counts of all `j` that give the best length; a strictly better length resets the count.

**Brute force →** Enumerate subsequences → O(2ⁿ). Length + count DP → O(n²).

**Steps**
1. `length[i] = count[i] = 1`.
2. For `j < i` with `nums[j] < nums[i]`: longer → replace length and count; equal → add the count.
3. Find the maximum length.
4. Sum the counts of indices with that length.

**Complexity**

- **Time:** O(n²).
- **Space:** O(n).

In [22]:
def find_number_of_lis(nums: list[int]) -> int:
    """Number of longest increasing subsequences."""
    n = len(nums)
    # length / count of the LIS ending exactly at i
    length, count = [1] * n, [1] * n
    for i in range(n):
        for j in range(i):
            if nums[j] < nums[i]:
                # a strictly longer chain through j
                if length[j] + 1 > length[i]:
                    length[i], count[i] = length[j] + 1, count[j]
                # another chain of the same best length
                elif length[j] + 1 == length[i]:
                    count[i] += count[j]
    best = max(length)
    return sum(c for l, c in zip(length, count) if l == best)


assert find_number_of_lis([1, 3, 5, 4, 7]) == 2
assert find_number_of_lis([2, 2, 2, 2, 2]) == 5
print(find_number_of_lis([1, 3, 5, 4, 7]))

2


### 1.4.3 [354. Russian Doll Envelopes](https://leetcode.com/problems/russian-doll-envelopes/) — ★ · Hard

**Problem.** Max number of envelopes that nest (both width and height strictly larger).  
**Signal.** Sort by width ascending, height *descending*, then LIS on heights.  
*New here: Reducing a 2D chain to 1D LIS with a clever sort.*

**Fine print.** Ask before coding:
- Must both width and height be strictly larger? → Yes.
- Can envelopes rotate? → No.

> 💡 **Key insight.** After sorting by width, an increasing run of heights is a nesting chain — provided equal widths can't chain, which sorting their heights *descending* guarantees.

**Brute force →** O(n²) DP over pairs → too slow for 10⁵. Sort + LIS → O(n log n).

**Steps**
1. Sort by `(width, −height)`.
2. Take the heights.
3. Return the LIS length of the heights.

**Complexity**

- **Time:** O(n log n).
- **Space:** O(n).

In [23]:
def max_envelopes(envelopes: list[list[int]]) -> int:
    """Longest nesting chain of envelopes."""
    # equal widths with descending heights can never both join the LIS
    envelopes = sorted(envelopes, key=lambda e: (e[0], -e[1]))
    return length_of_lis([h for _, h in envelopes])


assert max_envelopes([[5, 4], [6, 4], [6, 7], [2, 3]]) == 3
assert max_envelopes([[1, 1], [1, 1], [1, 1]]) == 1
print(max_envelopes([[5, 4], [6, 4], [6, 7], [2, 3]]))

3


### 1.4.4 [368. Largest Divisible Subset](https://leetcode.com/problems/largest-divisible-subset/) — ★ · Medium

**Problem.** Largest subset where every pair `(a, b)` has `a % b == 0` or `b % a == 0`.  
**Signal.** Sort, then LIS where "increasing" means "divisible by the previous".  
*New here: Reconstructing the actual sequence with parent pointers.*

**Fine print.** Ask before coding:
- Distinct positive values? → Yes.
- Several answers? → Return any.

> 💡 **Key insight.** After sorting, a chain where each element divides the next is pairwise divisible (divisibility is transitive), so it is LIS with "divides" instead of "<".

**Brute force →** Check every subset → O(2ⁿ · n²). Chain DP → O(n²).

**Steps**
1. Sort.
2. `dp[i]` = longest chain ending at `i`; `parent[i]` = previous element.
3. For `j < i` with `nums[i] % nums[j] == 0`, extend if longer.
4. Walk parent pointers back from the best `i`.

**Complexity**

- **Time:** O(n²).
- **Space:** O(n).

In [24]:
def largest_divisible_subset(nums: list[int]) -> list[int]:
    """A largest pairwise-divisible subset."""
    nums = sorted(nums)
    n = len(nums)
    dp, parent = [1] * n, [-1] * n
    for i in range(n):
        for j in range(i):
            # nums[j] divides nums[i]: extend j's chain
            if nums[i] % nums[j] == 0 and dp[j] + 1 > dp[i]:
                dp[i], parent[i] = dp[j] + 1, j
    # rebuild the chain from its last element
    i = dp.index(max(dp))
    out = []
    while i != -1:
        out.append(nums[i])
        i = parent[i]
    return out[::-1]


assert largest_divisible_subset([1, 2, 3]) in ([1, 2], [1, 3])
assert largest_divisible_subset([1, 2, 4, 8]) == [1, 2, 4, 8]
print(largest_divisible_subset([1, 2, 4, 8]))

[1, 2, 4, 8]


### 1.4.5 [1027. Longest Arithmetic Subsequence](https://leetcode.com/problems/longest-arithmetic-subsequence/) — ★ · Medium

**Problem.** Length of the longest arithmetic subsequence.  
**Signal.** State must include the common difference: `dp[i][d]`.  
*New here: Adding a second state dimension (the difference) stored in a dict.*

**Fine print.** Ask before coding:
- Can the difference be negative or 0? → Yes.
- Length at least 2? → Yes, any pair counts.

> 💡 **Key insight.** An arithmetic subsequence ending at `i` is determined by its difference, so the state is `(i, d)`; extending from `j` uses `d = nums[i] − nums[j]`.

**Brute force →** Every pair as the first two terms, then scan → O(n³). `dp[i][d]` → O(n²).

**Steps**
1. `dp[i]` = dict from difference to length ending at `i`.
2. For each `j < i`, `d = nums[i] − nums[j]`.
3. `dp[i][d] = dp[j].get(d, 1) + 1`.
4. Track the maximum.

**Complexity**

- **Time:** O(n²).
- **Space:** O(n²).

In [25]:
def longest_arith_seq_length(nums: list[int]) -> int:
    """Longest arithmetic subsequence length."""
    # dp[i][d] = longest arithmetic subsequence ending at i with difference d
    dp: list[dict[int, int]] = [{} for _ in nums]
    best = 2
    for i in range(len(nums)):
        for j in range(i):
            d = nums[i] - nums[j]
            # extend j's sequence with difference d (or start the pair j, i)
            dp[i][d] = dp[j].get(d, 1) + 1
            best = max(best, dp[i][d])
    return best


assert longest_arith_seq_length([3, 6, 9, 12]) == 4
assert longest_arith_seq_length([9, 4, 7, 2, 10]) == 3
assert longest_arith_seq_length([20, 1, 15, 3, 10, 5, 8]) == 4
print(longest_arith_seq_length([20, 1, 15, 3, 10, 5, 8]))

4


---
# 2) Knapsack

**In plain English.** Knapsack problems pick items to fill a capacity (a target sum, an
amount, a budget). The DP table is indexed by capacity: `dp[c]` = best / number of ways /
reachable for capacity `c`.

> **Analogy.** Making change at a till: to know the fewest coins for 11, look at the
> fewest coins for 11 − 1, 11 − 2 and 11 − 5 — one more coin on top of the best of those.

| Variant | Loop order | Why |
|---|---|---|
| **0/1** (each item once) | items outer, capacity **downwards** | reading `dp[c − w]` sees the value *before* this item was added |
| **unbounded** (reuse allowed) | capacity **upwards** | `dp[c − w]` may already include this item, which is allowed |
| count **combinations** | items outer, capacity inner | each combination is built in one fixed item order |
| count **ordered sequences** | capacity outer, items inner | every last item is tried at every capacity |

**Recognition signals**

- "coins", "amount", "target sum", "fewest items to reach"
- "can the array be split into two equal halves"
- "assign + / − to reach a target" (rewrite as a subset sum)
- two budgets at once (zeros and ones) → 2D capacity

**Common bugs**

- Iterating capacity upwards in a 0/1 problem → items reused.
- Swapping loops in counting problems → combinations vs. permutations mixed up.
- `dp[0]` wrong: 1 for counting (one way to make 0), 0 for minimum count, `True` for
  reachability.
- Using greedy for coin change (fails on coins `[1, 3, 4]`, amount 6).

**Don't confuse with**

| Pattern | Difference |
|---|---|
| Backtracking (Combination Sum) | *lists* every combination; knapsack DP *counts* or optimises them |
| Greedy | works only for special coin systems; DP always works |
| Subset-sum via bitset | the same 0/1 knapsack, using a Python int as the boolean array |

**Learning path.** Unbounded minimum (Coin Change) → unbounded counting combinations
(Coin Change II) → counting ordered sequences (Combination Sum IV) → 0/1 reachability
(Partition) → algebra to a subset sum (Target Sum, Last Stone Weight II) → two capacities
(Ones and Zeroes).

## 2.1 Subset sums and coins

Name the capacity, the item weight and what `dp[c]` stores; then pick the loop order from
the table above. That choice is the whole difference between the questions here.

### 2.1.1 [322. Coin Change](https://leetcode.com/problems/coin-change/) — ★★★ · Medium

**Problem.** Fewest coins summing to `amount` (unlimited coins), or −1.  
**Signal.** Unbounded knapsack minimising count.  
*New here: The capacity-indexed DP: `dp[a]` over amounts.*

**Fine print.** Ask before coding:
- Unlimited coins of each type? → Yes.
- `amount = 0`? → 0 coins.
- Impossible? → −1.

> 💡 **Key insight.** The last coin used is some `c`, so the fewest coins for `a` is one more than the fewest for `a − c`, minimised over coins.

**Brute force →** Recursion over every coin → exponential. Greedy (largest coin first) is wrong for `[1, 3, 4]`, amount 6. DP → O(amount · coins).

**Steps**
1. `dp[0] = 0`, everything else "infinity" (`amount + 1`).
2. For each amount `a`, for each coin `c ≤ a`: `dp[a] = min(dp[a], dp[a − c] + 1)`.
3. Return `dp[amount]`, or −1 if it is still infinity.

**Complexity**

- **Time:** O(amount · coins).
- **Space:** O(amount).

In [26]:
def coin_change(coins: list[int], amount: int) -> int:
    """Fewest coins for amount, or -1."""
    # amount + 1 coins is impossible, so it acts as infinity
    inf = amount + 1
    dp = [0] + [inf] * amount
    for a in range(1, amount + 1):
        # try each coin as the last one used
        for c in coins:
            if c <= a:
                dp[a] = min(dp[a], dp[a - c] + 1)
    return dp[amount] if dp[amount] < inf else -1


assert coin_change([1, 2, 5], 11) == 3
assert coin_change([2], 3) == -1
assert coin_change([1], 0) == 0
assert coin_change([1, 3, 4], 6) == 2
print(coin_change([1, 2, 5], 11))

3


### 2.1.2 [518. Coin Change II](https://leetcode.com/problems/coin-change-ii/) — ★★ · Medium

**Problem.** Number of *combinations* of coins that make `amount`.  
**Signal.** Unbounded knapsack counting; coins in the outer loop so each combination counts once.  
*New here: Loop order decides combinations vs. permutations.*

**Fine print.** Ask before coding:
- Combinations, not orderings? → Yes, which fixes the loop order (coins outer).
- `amount = 0`? → 1 way.

> 💡 **Key insight.** Processing coins one at a time means every combination is built with its coins in a fixed order, so `1 + 2` and `2 + 1` are counted once.

**Brute force →** Backtracking that lists combinations → exponential. Counting DP → O(amount · coins).

**Steps**
1. `dp[0] = 1` (one way to make 0: no coins).
2. For each coin, for `a` from `coin` up to `amount`: `dp[a] += dp[a − coin]`.
3. Return `dp[amount]`.

**Complexity**

- **Time:** O(amount · coins).
- **Space:** O(amount).

⚠️ **Trap.** Swapping the loops counts ordered sequences instead (that is Combination Sum IV).

In [27]:
def change(amount: int, coins: list[int]) -> int:
    """Number of coin combinations for amount."""
    dp = [1] + [0] * amount
    # coins outer: each combination counted in one coin order
    for c in coins:
        # upwards: the same coin may be used again
        for a in range(c, amount + 1):
            dp[a] += dp[a - c]
    return dp[amount]


assert change(5, [1, 2, 5]) == 4
assert change(3, [2]) == 0
assert change(10, [10]) == 1
print(change(5, [1, 2, 5]))

4


### 2.1.3 [377. Combination Sum IV](https://leetcode.com/problems/combination-sum-iv/) — ★★★ · Medium

**Problem.** Number of *ordered* sequences of `nums` (reuse allowed) summing to `target`.  
**Signal.** Counting sequences → target in the outer loop.  
*New here: The other loop order: counting permutations.*

**Fine print.** Ask before coding:
- Ordered sequences? → Yes, `(1,2)` and `(2,1)` both count.
- Follow-up: negative numbers? → You would need a length limit, or the count is infinite.

> 💡 **Key insight.** An ordered sequence summing to `t` ends with some `x`; everything before it is any sequence summing to `t − x`.

**Brute force →** Backtracking over sequences → exponential. DP → O(target · n).

**Steps**
1. `dp[0] = 1`.
2. For each `t` from 1 to target: `dp[t] = Σ dp[t − x]` over `x ≤ t`.
3. Return `dp[target]`.

**Complexity**

- **Time:** O(target · n).
- **Space:** O(target).

In [28]:
def combination_sum4(nums: list[int], target: int) -> int:
    """Ordered ways to sum to target."""
    dp = [1] + [0] * target
    # target outer: every number can be the last element at every total
    for t in range(1, target + 1):
        dp[t] = sum(dp[t - x] for x in nums if x <= t)
    return dp[target]


assert combination_sum4([1, 2, 3], 4) == 7
assert combination_sum4([9], 3) == 0
print(combination_sum4([1, 2, 3], 4))

7


### 2.1.4 [416. Partition Equal Subset Sum](https://leetcode.com/problems/partition-equal-subset-sum/) — ★★ · Medium

**Problem.** Can the array be split into two subsets with equal sums?  
**Signal.** Subset sum to `total / 2` → 0/1 knapsack on booleans.  
*New here: 0/1 knapsack: capacity iterated *downwards*.*

**Fine print.** Ask before coding:
- Positive values? → Yes.
- Must every element be used? → Yes, every element goes into one of the two subsets.

> 💡 **Key insight.** Two equal halves exist iff some subset sums to `total / 2`; iterating capacity downwards guarantees each number is used at most once.

**Brute force →** Try all 2ⁿ subsets → exponential. Boolean DP → O(n · total).

**Steps**
1. Odd total → `False`.
2. `dp[0] = True`; `dp[s]` = some subset sums to `s`.
3. For each number `x`, for `s` from `half` down to `x`: `dp[s] |= dp[s − x]`.
4. Return `dp[half]`.

**Complexity**

- **Time:** O(n · total).
- **Space:** O(total).

In [29]:
def can_partition(nums: list[int]) -> bool:
    """True if nums splits into two equal-sum halves."""
    total = sum(nums)
    if total % 2:
        return False
    half = total // 2
    # dp[s]: some subset of the numbers seen so far sums to s
    dp = [True] + [False] * half
    for x in nums:
        # downwards so x is used at most once
        for s in range(half, x - 1, -1):
            dp[s] = dp[s] or dp[s - x]
    return dp[half]


assert can_partition([1, 5, 11, 5])
assert not can_partition([1, 2, 3, 5])
print(can_partition([1, 5, 11, 5]))

True


### 2.1.5 [494. Target Sum](https://leetcode.com/problems/target-sum/) — ★★ · Medium

**Problem.** Assign `+` or `−` to each number; count assignments evaluating to `target`.  
**Signal.** Split into positive set `P` and negative set `N`: `P = (total + target) / 2` → count subsets.  
*New here: Algebra that turns a ± problem into a plain subset sum.*

**Fine print.** Ask before coding:
- Must each number get a sign? → Yes.
- Can values be 0? → Yes, and each 0 doubles the count.

> 💡 **Key insight.** From `P − N = target` and `P + N = total`, the positive set must sum to `(total + target) / 2`, so count subsets with that sum.

**Brute force →** Try all 2ⁿ sign patterns → exponential. Subset-count DP → O(n · total).

**Steps**
1. If `|target| > total` or `total + target` is odd, return 0.
2. `goal = (total + target) / 2`; `dp[0] = 1`.
3. For each number, for `s` from `goal` down to `x`: `dp[s] += dp[s − x]`.
4. Return `dp[goal]`.

**Complexity**

- **Time:** O(n · total).
- **Space:** O(total).

In [30]:
def find_target_sum_ways(nums: list[int], target: int) -> int:
    """Number of +/- assignments reaching target."""
    total = sum(nums)
    # P must be a non-negative integer
    if abs(target) > total or (total + target) % 2:
        return 0
    goal = (total + target) // 2
    # dp[s] = number of subsets summing to s
    dp = [1] + [0] * goal
    for x in nums:
        for s in range(goal, x - 1, -1):
            dp[s] += dp[s - x]
    return dp[goal]


assert find_target_sum_ways([1, 1, 1, 1, 1], 3) == 5
assert find_target_sum_ways([1], 1) == 1
assert find_target_sum_ways([0, 0, 1], 1) == 4
print(find_target_sum_ways([1, 1, 1, 1, 1], 3))

5


### 2.1.6 [1049. Last Stone Weight II](https://leetcode.com/problems/last-stone-weight-ii/) — ★ · Medium

**Problem.** Smash stones pairwise (result `|a − b|`); minimise the last remaining weight.  
**Signal.** The result is `|sum(A) − sum(B)|` for some split → subset sum closest to `total / 2`.  
*New here: Recognising a hidden partition problem.*

**Fine print.** Ask before coding:
- Can 0 remain? → Yes.
- Do you return the final weight? → Yes, the smallest possible.

> 💡 **Key insight.** Every smashing order ends with some stones counted `+` and others `−`, so the last weight is the difference of two groups; make one group as close to half as possible.

**Brute force →** Simulate every smashing order → factorial. Reachable sums → O(n · total).

**Steps**
1. Build the set of reachable subset sums up to `total // 2`.
2. Take the largest reachable `s`.
3. Answer = `total − 2s`.

**Complexity**

- **Time:** O(n · total).
- **Space:** O(total).

In [31]:
def last_stone_weight_ii(stones: list[int]) -> int:
    """Smallest possible final stone weight."""
    total = sum(stones)
    # subset sums reachable without passing half the total
    reach = {0}
    for x in stones:
        reach |= {s + x for s in reach if s + x <= total // 2}
    # two groups: s and total - s
    return total - 2 * max(reach)


assert last_stone_weight_ii([2, 7, 4, 1, 8, 1]) == 1
assert last_stone_weight_ii([31, 26, 33, 21, 40]) == 5
print(last_stone_weight_ii([2, 7, 4, 1, 8, 1]))

1


### 2.1.7 [474. Ones and Zeroes](https://leetcode.com/problems/ones-and-zeroes/) — ★ · Medium

**Problem.** Largest subset of binary strings using at most `m` zeros and `n` ones in total.  
**Signal.** 0/1 knapsack with two capacities.  
*New here: Two capacities → a 2D table, both iterated downwards.*

**Fine print.** Ask before coding:
- At most `m` zeros and `n` ones in total? → Yes.
- Count strings, not characters? → Yes.

> 💡 **Key insight.** Each string costs some zeros and some ones; it is 0/1 knapsack where the "weight" has two components.

**Brute force →** Try all subsets → O(2^L). 2D knapsack → O(L · m · n).

**Steps**
1. `dp[i][j]` = most strings using ≤ i zeros and ≤ j ones.
2. For each string, count its zeros `z` and ones `o`.
3. For `i` from m down to z and `j` from n down to o: `dp[i][j] = max(dp[i][j], dp[i − z][j − o] + 1)`.
4. Return `dp[m][n]`.

**Complexity**

- **Time:** O(L · m · n).
- **Space:** O(m · n).

In [32]:
def find_max_form(strs: list[str], m: int, n: int) -> int:
    """Largest subset within the zero/one budget."""
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for s in strs:
        zeros, ones = s.count("0"), s.count("1")
        # both budgets downwards: each string used at most once
        for i in range(m, zeros - 1, -1):
            for j in range(n, ones - 1, -1):
                dp[i][j] = max(dp[i][j], dp[i - zeros][j - ones] + 1)
    return dp[m][n]


assert find_max_form(["10", "0001", "111001", "1", "0"], 5, 3) == 4
assert find_max_form(["10", "0", "1"], 1, 1) == 2
print(find_max_form(["10", "0001", "111001", "1", "0"], 5, 3))

4


---
# 3) State Machine DP

**In plain English.** When what you may do today depends on a *mode* (holding a share,
cooling down, how many trades used), keep one best value per mode and update all modes
each day.

> **Analogy.** A traffic light: the allowed next colour depends only on the current
> colour. Draw the modes as boxes and the allowed changes as arrows — each arrow is one
> line of code.

**Recognition signals**

- buy / sell stock with extra rules (cooldown, fee, at most k trades)
- "you cannot do X right after Y"
- a small number of modes that persist between steps

**Common bugs**

- Updating one mode and then using the *new* value for another mode in the same step
  (use simultaneous assignment or temporaries when the order matters).
- Initialising "holding" to 0 instead of −∞ (you can't hold before buying).
- Forgetting that "at most k" allows fewer trades.

**Don't confuse with**

| Pattern | Difference |
|---|---|
| Greedy (Stock II) | unlimited trades without fees → just sum every rise |
| 1D DP | one value per index; state machines keep several values per index |
| Single-trade stock (running min) | only one transaction, no modes needed |

**Learning path.** Unlimited trades (greedy) → cooldown (three modes) → fee (two modes) →
two trades (four modes) → k trades (arrays of modes).

## 3.1 Stock problems

Modes: `hold` (own a share; best balance so far) and `cash` / `sold` / `rest` (own
nothing). Each day: new hold = max(keep holding, buy from a free mode); new cash =
max(keep cash, sell from hold).

### 3.1.1 [122. Best Time to Buy and Sell Stock II](https://leetcode.com/problems/best-time-to-buy-and-sell-stock-ii/) — ★★ · Medium

**Problem.** Unlimited transactions (hold at most one share); maximise profit.  
**Signal.** Two states (holding / not holding) — or greedily take every upward move.  
*New here: When the state machine collapses into a greedy rule.*

**Fine print.** Ask before coding:
- Can you buy and sell on the same day? → Yes, which adds nothing.
- At most one share at a time? → Yes.

> 💡 **Key insight.** Any profitable hold from day a to day b equals the sum of its daily rises, so collecting every positive daily rise is optimal.

**Brute force →** Try all buy/sell schedules → exponential. Two-state DP → O(n). Greedy sum → O(n), one line.

**Steps**
1. For each pair of consecutive days, compute the change.
2. Add it if positive.

**Complexity**

- **Time:** O(n).
- **Space:** O(1).

In [33]:
def max_profit_ii(prices: list[int]) -> int:
    """Profit with unlimited transactions."""
    # every rise can be captured by buying yesterday and selling today
    return sum(max(0, b - a) for a, b in zip(prices, prices[1:]))


assert max_profit_ii([7, 1, 5, 3, 6, 4]) == 7
assert max_profit_ii([1, 2, 3, 4, 5]) == 4
assert max_profit_ii([7, 6, 4, 3, 1]) == 0
print(max_profit_ii([7, 1, 5, 3, 6, 4]))

7


### 3.1.2 [309. Best Time to Buy and Sell Stock with Cooldown](https://leetcode.com/problems/best-time-to-buy-and-sell-stock-with-cooldown/) — ★★ · Medium

**Problem.** Unlimited transactions, but after selling you must wait one day before buying.  
**Signal.** Three states: `hold`, `sold` (just sold today), `rest` (free to buy).  
*New here: A third mode to encode a waiting rule.*

**Fine print.** Ask before coding:
- Does the cooldown apply only after selling? → Yes, buying has no cooldown.

> 💡 **Key insight.** Buying is only allowed from `rest`, and `sold` only becomes `rest` a day later — that delay *is* the cooldown.

**Brute force →** Recursion over buy/sell/wait each day → O(3ⁿ). Three rolling states → O(n).

**Steps**
1. `hold = −∞`, `sold = rest = 0`.
2. Each day, simultaneously: `hold = max(hold, rest − p)`, `sold = hold + p`, `rest = max(rest, sold)`.
3. Return `max(sold, rest)`.

**Complexity**

- **Time:** O(n).
- **Space:** O(1).

In [34]:
def max_profit_cooldown(prices: list[int]) -> int:
    """Profit with a one-day cooldown after selling."""
    # hold: own a share; sold: sold today; rest: free to buy
    hold, sold, rest = float("-inf"), 0, 0
    for p in prices:
        # right-hand sides all use yesterday's values
        hold, sold, rest = max(hold, rest - p), hold + p, max(rest, sold)
    return max(sold, rest)


assert max_profit_cooldown([1, 2, 3, 0, 2]) == 3
assert max_profit_cooldown([1]) == 0
print(max_profit_cooldown([1, 2, 3, 0, 2]))

3


### 3.1.3 [714. Best Time to Buy and Sell Stock with Transaction Fee](https://leetcode.com/problems/best-time-to-buy-and-sell-stock-with-transaction-fee/) — ★ · Medium

**Problem.** Unlimited transactions, each sale costs `fee`.  
**Signal.** Two states with the fee charged on selling.  
*New here: A cost attached to one transition.*

**Fine print.** Ask before coding:
- Is the fee paid once per full transaction? → Yes, charge it on the sell.

> 💡 **Key insight.** The fee is paid on the hold → cash arrow, so subtract it there and nowhere else.

**Brute force →** Greedy on rises fails because small rises don't cover the fee. Two states → O(n).

**Steps**
1. `cash = 0`, `hold = −prices[0]`.
2. Each day: `cash = max(cash, hold + p − fee)`.
3. `hold = max(hold, cash − p)`.
4. Return `cash`.

**Complexity**

- **Time:** O(n).
- **Space:** O(1).

In [35]:
def max_profit_fee(prices: list[int], fee: int) -> int:
    """Profit with a per-transaction fee."""
    cash, hold = 0, -prices[0]
    for p in prices[1:]:
        # sell today (paying the fee) or stay in cash
        cash = max(cash, hold + p - fee)
        # buy today or keep holding
        hold = max(hold, cash - p)
    return cash


assert max_profit_fee([1, 3, 2, 8, 4, 9], 2) == 8
assert max_profit_fee([1, 3, 7, 5, 10, 3], 3) == 6
print(max_profit_fee([1, 3, 2, 8, 4, 9], 2))

8


### 3.1.4 [123. Best Time to Buy and Sell Stock III](https://leetcode.com/problems/best-time-to-buy-and-sell-stock-iii/) — ★★ · Hard

**Problem.** At most two transactions.  
**Signal.** Four states: after first buy, first sell, second buy, second sell.  
*New here: A chain of modes for a limited number of trades.*

**Fine print.** Ask before coding:
- At most two transactions, possibly fewer? → Yes.
- No overlapping transactions? → Yes.

> 💡 **Key insight.** Each mode is "best balance after this action so far", and each feeds the next: buy₁ → sell₁ → buy₂ → sell₂.

**Brute force →** Split the array at every day and solve one trade on each side → O(n²) (O(n) with prefix/suffix arrays). Four states → O(n), O(1) space.

**Steps**
1. `b1 = b2 = −∞`, `s1 = s2 = 0`.
2. Each day: `b1 = max(b1, −p)`, `s1 = max(s1, b1 + p)`.
3. `b2 = max(b2, s1 − p)`, `s2 = max(s2, b2 + p)`.
4. Return `s2`.

**Complexity**

- **Time:** O(n).
- **Space:** O(1).

In [36]:
def max_profit_iii(prices: list[int]) -> int:
    """Profit with at most two transactions."""
    b1 = b2 = float("-inf")
    s1 = s2 = 0
    for p in prices:
        # each mode builds on the one before it
        b1 = max(b1, -p)
        s1 = max(s1, b1 + p)
        b2 = max(b2, s1 - p)
        s2 = max(s2, b2 + p)
    return s2


assert max_profit_iii([3, 3, 5, 0, 0, 3, 1, 4]) == 6
assert max_profit_iii([1, 2, 3, 4, 5]) == 4
assert max_profit_iii([7, 6, 4, 3, 1]) == 0
print(max_profit_iii([3, 3, 5, 0, 0, 3, 1, 4]))

6


### 3.1.5 [188. Best Time to Buy and Sell Stock IV](https://leetcode.com/problems/best-time-to-buy-and-sell-stock-iv/) — ★★ · Hard

**Problem.** At most `k` transactions.  
**Signal.** Generalise the four-state version to arrays `buy[1..k]`, `sell[1..k]`.  
*New here: Turning a fixed chain of modes into arrays.*

**Fine print.** Ask before coding:
- Can `k` be larger than `n / 2`? → Yes, then it is the same as unlimited (122).

> 💡 **Key insight.** Transaction `j`'s buy builds on transaction `j − 1`'s sell, exactly like the two-trade chain, repeated k times.

**Brute force →** `dp[j][i]` with an inner loop over buy days → O(k · n²). Rolling arrays → O(n · k).

**Steps**
1. If `k ≥ n / 2`, it is the unlimited case.
2. `buy[j] = −∞`, `sell[j] = 0`.
3. Each day, for `j` in 1..k: `buy[j] = max(buy[j], sell[j − 1] − p)`, `sell[j] = max(sell[j], buy[j] + p)`.
4. Return `sell[k]`.

**Complexity**

- **Time:** O(n · k).
- **Space:** O(k).

In [37]:
def max_profit_iv(k: int, prices: list[int]) -> int:
    """Profit with at most k transactions."""
    # enough trades to take every rise
    if k >= len(prices) // 2:
        return max_profit_ii(prices)
    buy = [float("-inf")] * (k + 1)
    sell = [0] * (k + 1)
    for p in prices:
        for j in range(1, k + 1):
            # j-th buy uses the profit of the (j - 1)-th sell
            buy[j] = max(buy[j], sell[j - 1] - p)
            sell[j] = max(sell[j], buy[j] + p)
    return sell[k]


assert max_profit_iv(2, [2, 4, 1]) == 2
assert max_profit_iv(2, [3, 2, 6, 5, 0, 3]) == 7
print(max_profit_iv(2, [3, 2, 6, 5, 0, 3]))

7


---
# 4) Grid DP

**In plain English.** On a grid where you only move right/down (or down), the best answer
for a cell comes from the cells you could have arrived from. Fill the grid in reading
order and each cell reads already-finished neighbours.

> **Analogy.** Counting routes through a city grid of one-way streets: the number of
> routes to a crossing is the routes to the crossing above plus the routes to the one on
> the left.

Because `dp[r][c]` only needs the row above and the cell to the left, **one row array**
updated in place is enough.

**Recognition signals**

- "robot moves only right or down", "falling path", "triangle"
- "minimum path sum", "number of paths", "with obstacles"
- "largest square of 1s"
- "minimum starting health" → DP *backwards* from the goal
- 4-direction moves with a strict increase → memoised DFS (no cycles possible)

**Common bugs**

- First row / column handled like the interior (they have one predecessor).
- Forward DP when the constraint depends on the future (Dungeon Game) — go backwards.
- Forgetting the saved diagonal value when compressing to one row (Maximal Square).

**Don't confuse with**

| Pattern | Difference |
|---|---|
| Grid BFS | any direction, shortest steps; DP needs an acyclic move order |
| Dijkstra on a grid | arbitrary directions with costs; DP works only when moves can't loop |
| Memoised DFS | the same DP when the order is implied by values, not by position |

**Learning path.** Count paths → with obstacles → minimum sum → triangle (bottom-up) →
three predecessors (falling path) → squares (min of three neighbours) → backwards DP
(Dungeon) → DP on a value-ordered DAG (Longest Increasing Path).

## 4.1 Paths on grids

For each cell, list where you could have come from; the transition combines those cells
(`+` for counting, `min` for cost). Then decide the fill order and compress to one row.

### 4.1.1 [62. Unique Paths](https://leetcode.com/problems/unique-paths/) — ★★★ · Medium

**Problem.** Robot moves only right or down in an `m × n` grid; number of paths from top-left to bottom-right.  
**Signal.** `dp[r][c] = dp[r−1][c] + dp[r][c−1]`.  
*New here: 2D DP compressed into one row.*

**Fine print.** Ask before coding:
- Moves are right or down only? → Yes.
- Does the answer fit in 32 bits? → Yes, ≤ 2·10⁹.

> 💡 **Key insight.** Every path into a cell arrives from above or from the left, so the counts add.

**Brute force →** Enumerate paths → C(m+n−2, m−1), exponential. DP → O(mn). (Closed form: that binomial.)

**Steps**
1. The first row is all 1s.
2. For every next row, `row[c] += row[c − 1]` (above + left).
3. Return `row[-1]`.

**Complexity**

- **Time:** O(mn).
- **Space:** O(n).

In [38]:
def unique_paths(m: int, n: int) -> int:
    """Lattice paths in an m x n grid."""
    # top row: one path to each cell
    row = [1] * n
    for _ in range(1, m):
        for c in range(1, n):
            # row[c] still holds the value from above; add the left cell
            row[c] += row[c - 1]
    return row[-1]


assert unique_paths(3, 7) == 28
assert unique_paths(3, 2) == 3
print(unique_paths(3, 7))

28


### 4.1.2 [63. Unique Paths II](https://leetcode.com/problems/unique-paths-ii/) — ★★ · Medium

**Problem.** Same with obstacles (`1` cells cannot be entered).  
**Signal.** Unique Paths with `dp = 0` on obstacles.  
*New here: Blocked cells contribute 0.*

**Fine print.** Ask before coding:
- Can the start or end cell be an obstacle? → Yes, then the answer is 0.

> 💡 **Key insight.** An obstacle has zero paths into it, which automatically cuts off every path through it.

**Brute force →** DFS over paths → exponential. DP → O(mn).

**Steps**
1. `row[0] = 1`, the rest 0.
2. For each cell: obstacle → 0; otherwise add the left value (column 0 keeps the value from above).
3. Return `row[-1]`.

**Complexity**

- **Time:** O(mn).
- **Space:** O(n).

In [39]:
def unique_paths_with_obstacles(grid: list[list[int]]) -> int:
    """Lattice paths avoiding obstacles."""
    row = [0] * len(grid[0])
    row[0] = 1
    for r in grid:
        for c, cell in enumerate(r):
            # no path enters an obstacle
            if cell:
                row[c] = 0
            # from above (already in row[c]) plus from the left
            elif c:
                row[c] += row[c - 1]
    return row[-1]


assert unique_paths_with_obstacles([[0, 0, 0], [0, 1, 0], [0, 0, 0]]) == 2
assert unique_paths_with_obstacles([[0, 1], [0, 0]]) == 1
assert unique_paths_with_obstacles([[1]]) == 0
print(unique_paths_with_obstacles([[0, 0, 0], [0, 1, 0], [0, 0, 0]]))

2


### 4.1.3 [64. Minimum Path Sum](https://leetcode.com/problems/minimum-path-sum/) — ★★ · Medium

**Problem.** Minimum sum path from top-left to bottom-right moving right/down.  
**Signal.** `dp[r][c] = grid[r][c] + min(up, left)`.  
*New here: Minimising cost instead of counting.*

**Fine print.** Ask before coding:
- Non-negative values? → Yes.
- Can I modify the grid? → Usually yes, which gives O(1) extra space.

> 💡 **Key insight.** The cheapest way into a cell is its own value plus the cheaper of its two predecessors.

**Brute force →** All paths → exponential. DP → O(mn).

**Steps**
1. `row` starts as ∞ except `row[0] = 0`.
2. For each cell: `row[c] = cell + min(row[c] (above), row[c − 1] (left))`.
3. Return `row[-1]`.

**Complexity**

- **Time:** O(mn).
- **Space:** O(n).

In [40]:
def min_path_sum(grid: list[list[int]]) -> int:
    """Cheapest monotone path sum."""
    inf = float("inf")
    # infinity outside the grid; 0 "above" the start
    row = [inf] * len(grid[0])
    row[0] = 0
    for r in grid:
        for c, cell in enumerate(r):
            left = row[c - 1] if c else inf
            # row[c] is the cell above
            row[c] = cell + min(row[c], left)
    return row[-1]


assert min_path_sum([[1, 3, 1], [1, 5, 1], [4, 2, 1]]) == 7
assert min_path_sum([[1, 2, 3], [4, 5, 6]]) == 12
print(min_path_sum([[1, 3, 1], [1, 5, 1], [4, 2, 1]]))

7


### 4.1.4 [120. Triangle](https://leetcode.com/problems/triangle/) — ★★ · Medium

**Problem.** Minimum path sum from top to bottom of a triangle (move to adjacent numbers below).  
**Signal.** Bottom-up: each cell picks the better of its two children.  
*New here: Choosing the fill direction that removes edge cases.*

**Fine print.** Ask before coding:
- Adjacent means index `i` or `i + 1` in the next row? → Yes.
- O(n) extra space? → That is the follow-up.

> 💡 **Key insight.** Going bottom-up, every cell has exactly two children below, so no border checks are needed and the answer ends up at the top.

**Brute force →** All 2ⁿ⁻¹ paths. Top-down DP needs border cases. Bottom-up → O(n²).

**Steps**
1. Start with a copy of the last row.
2. For each row upward: `dp[c] = tri[r][c] + min(dp[c], dp[c + 1])`.
3. Return `dp[0]`.

**Complexity**

- **Time:** O(n²).
- **Space:** O(n).

In [41]:
def minimum_total(triangle: list[list[int]]) -> int:
    """Min top-to-bottom path sum."""
    # best path sums starting at each cell of the bottom row
    dp = triangle[-1].copy()
    for r in range(len(triangle) - 2, -1, -1):
        for c in range(r + 1):
            # step to the better of the two cells below
            dp[c] = triangle[r][c] + min(dp[c], dp[c + 1])
    return dp[0]


assert minimum_total([[2], [3, 4], [6, 5, 7], [4, 1, 8, 3]]) == 11
assert minimum_total([[-10]]) == -10
print(minimum_total([[2], [3, 4], [6, 5, 7], [4, 1, 8, 3]]))

11


### 4.1.5 [931. Minimum Falling Path Sum](https://leetcode.com/problems/minimum-falling-path-sum/) — ★ · Medium

**Problem.** Path from any top cell to any bottom cell, moving down, down-left or down-right; minimum sum?  
**Signal.** Each cell adds the best of three cells above.  
*New here: Three predecessors and any start / any end.*

**Fine print.** Ask before coding:
- Square matrix? → Yes.
- Can you start in any top cell? → Yes.

> 💡 **Key insight.** The best falling path into a cell is its value plus the best of the (up to) three cells above it.

**Brute force →** All paths → O(n · 3ⁿ). Row DP → O(n²).

**Steps**
1. `prev` = first row.
2. For each next row, `new[c] = row[c] + min(prev[c − 1 .. c + 1])` within bounds.
3. Return `min(prev)` after the last row.

**Complexity**

- **Time:** O(n²).
- **Space:** O(n).

In [42]:
def min_falling_path_sum(matrix: list[list[int]]) -> int:
    """Min falling path sum."""
    prev = matrix[0]
    n = len(prev)
    for row in matrix[1:]:
        # best of up-left, up, up-right (clipped at the borders)
        prev = [row[c] + min(prev[max(0, c - 1) : min(n, c + 2)]) for c in range(n)]
    # the path may end in any column
    return min(prev)


assert min_falling_path_sum([[2, 1, 3], [6, 5, 4], [7, 8, 9]]) == 13
assert min_falling_path_sum([[-19, 57], [-40, -5]]) == -59
print(min_falling_path_sum([[2, 1, 3], [6, 5, 4], [7, 8, 9]]))

13


### 4.1.6 [221. Maximal Square](https://leetcode.com/problems/maximal-square/) — ★★ · Medium

**Problem.** Area of the largest square of `"1"`s.  
**Signal.** `dp[r][c]` = side of the largest square with bottom-right corner at `(r, c)`.  
*New here: A state defined by a corner, and keeping the diagonal while compressing.*

**Fine print.** Ask before coding:
- Return the area, not the side? → The area.
- Cells are the characters `"1"`/`"0"`? → Yes.

> 💡 **Key insight.** A square of side k ending at `(r, c)` needs squares of side k − 1 ending above, to the left and diagonally, so the side is `1 + min` of those three.

**Brute force →** Try every top-left corner and size → O((mn)²). DP → O(mn).

**Steps**
1. One row array with a leading 0, plus a `diag` variable.
2. For each `"1"` cell: `dp[c] = 1 + min(above, left, diag)`; for `"0"`: 0.
3. Save the old `dp[c]` as the next `diag`.
4. Track the largest side; return its square.

**Complexity**

- **Time:** O(mn).
- **Space:** O(n).

In [43]:
def maximal_square(matrix: list[list[str]]) -> int:
    """Area of the largest all-ones square."""
    cols = len(matrix[0])
    # dp[c] = largest square side ending at (current row, c - 1)
    dp = [0] * (cols + 1)
    best = 0
    for row in matrix:
        # value up-left of the current cell
        diag = 0
        for c in range(1, cols + 1):
            above = dp[c]
            # limited by the smallest of the three neighbouring squares
            dp[c] = 1 + min(dp[c], dp[c - 1], diag) if row[c - 1] == "1" else 0
            diag = above
            best = max(best, dp[c])
    return best * best


m = [list("10100"), list("10111"), list("11111"), list("10010")]
assert maximal_square(m) == 4
assert maximal_square([list("01"), list("10")]) == 1
print(maximal_square(m))

4


### 4.1.7 [174. Dungeon Game](https://leetcode.com/problems/dungeon-game/) — ★ · Hard

**Problem.** Knight moves right/down from top-left to bottom-right; cells add or subtract health; health must stay ≥ 1. Minimum starting health?  
**Signal.** Forward DP can't decide; go **backwards** from the princess.  
*New here: Backward DP because the constraint depends on what comes later.*

**Fine print.** Ask before coding:
- Must health stay ≥ 1 at every step, including the start and end cells? → Yes.
- Do cells apply their effect when entered? → Yes.

> 💡 **Key insight.** The health you need entering a cell depends only on the cells after it, so fill from the goal: need = `max(1, min(need of next cells) − cell value)`.

**Brute force →** Binary search on the starting health with a forward check → O(mn log H). Backward DP → O(mn).

**Steps**
1. Beyond the princess, the requirement is 1 (one virtual cell).
2. Fill rows from the bottom, columns from the right.
3. `need = max(1, min(right, down) − dungeon[r][c])`.
4. Return the value at the start.

**Complexity**

- **Time:** O(mn).
- **Space:** O(n).

⚠️ **Trap.** Forward DP fails: the best path so far may leave too little health for what comes later.

In [44]:
def calculate_minimum_hp(dungeon: list[list[int]]) -> int:
    """Minimum initial health to survive."""
    rows, cols = len(dungeon), len(dungeon[0])
    inf = float("inf")
    # need[c] = health needed entering (row below, c); 1 just past the goal
    need = [inf] * (cols + 1)
    need[cols - 1] = 1
    for r in range(rows - 1, -1, -1):
        nxt = [inf] * (cols + 1)
        for c in range(cols - 1, -1, -1):
            # cheaper of going down or right
            best = min(need[c], nxt[c + 1])
            # health must stay at least 1 after this cell
            nxt[c] = max(1, best - dungeon[r][c])
        need = nxt
    return need[0]


assert calculate_minimum_hp([[-2, -3, 3], [-5, -10, 1], [10, 30, -5]]) == 7
assert calculate_minimum_hp([[0]]) == 1
print(calculate_minimum_hp([[-2, -3, 3], [-5, -10, 1], [10, 30, -5]]))

7


### 4.1.8 [329. Longest Increasing Path in a Matrix](https://leetcode.com/problems/longest-increasing-path-in-a-matrix/) — ★★ · Hard

**Problem.** Length of the longest strictly increasing path (4 directions) in a matrix.  
**Signal.** DFS + memo: the increasing constraint makes the graph acyclic.  
*New here: Memoised DFS on a DAG whose order comes from values.*

**Fine print.** Ask before coding:
- Strictly increasing? → Yes.
- 4 directions, no wrap? → Yes.

> 💡 **Key insight.** A strictly increasing path can never revisit a cell, so the "move to a larger neighbour" graph is acyclic and each cell's answer can be cached.

**Brute force →** DFS from every cell without memo → exponential. Memoised DFS → O(mn).

**Steps**
1. `longest(r, c)` = 1 + max of `longest` over larger neighbours.
2. Cache it with `@cache`.
3. Return the max over all cells.

**Complexity**

- **Time:** O(mn) — each cell computed once, 4 neighbours each.
- **Space:** O(mn) — the cache and recursion.

In [45]:
def longest_increasing_path(matrix: list[list[int]]) -> int:
    """Longest strictly increasing path length."""
    rows, cols = len(matrix), len(matrix[0])

    @cache
    def longest(r: int, c: int) -> int:
        """Longest increasing path starting at (r, c)."""
        best = 1
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            nr, nc = r + dr, c + dc
            # only climbing moves: no cycles, so no visited set
            if 0 <= nr < rows and 0 <= nc < cols and matrix[nr][nc] > matrix[r][c]:
                best = max(best, 1 + longest(nr, nc))
        return best

    return max(longest(r, c) for r in range(rows) for c in range(cols))


assert longest_increasing_path([[9, 9, 4], [6, 6, 8], [2, 1, 1]]) == 4
assert longest_increasing_path([[3, 4, 5], [3, 2, 6], [2, 2, 1]]) == 4
print(longest_increasing_path([[9, 9, 4], [6, 6, 8], [2, 1, 1]]))

4


---
# 5) String DP

**In plain English.** String DP compares prefixes (two strings) or substrings (one string).
For two strings, `dp[i][j]` answers the question for `a[:i]` and `b[:j]`, built from what
happens to the *last* characters.

> **Analogy.** Comparing two drafts of an essay line by line: if the last lines match,
> they cost nothing; if not, you either delete a line from one draft, insert one, or edit
> it — and the rest is the same comparison on shorter drafts.

**Two strings:** if the last characters match, extend `dp[i−1][j−1]`; otherwise combine
`dp[i−1][j]` (drop from `a`) and `dp[i][j−1]` (drop from `b`). Row `0` / column `0` are
the empty-prefix base cases. **One string, palindromes:** expand around centres, or
`dp[l][r]` over intervals filled by increasing length.

**Recognition signals**

- "longest common subsequence", "edit distance", "delete to make equal"
- "is s3 an interleaving of s1 and s2", "number of distinct subsequences"
- pattern matching with `.`, `?`, `*`
- "palindromic substring / subsequence / partition"

**Common bugs**

- Off-by-one between `dp[i][j]` (prefix lengths) and string indices `a[i − 1]`.
- Losing the diagonal value when compressing to one row (save it before overwriting).
- Wrong base row: edit distance has `dp[0][j] = j`, LCS has 0, matching has `True` only
  where the pattern can match empty.
- Subsequence vs substring: subsequences skip characters, substrings are contiguous.

**Don't confuse with**

| Pattern | Difference |
|---|---|
| Sliding window on strings | contiguous substrings with a validity rule; no alignment of two strings |
| KMP / substring search | exact contiguous match; DP handles gaps and edits |
| Interval DP | one sequence split into ranges; two-string DP aligns two sequences |

**Learning path.** Palindromes by centre expansion (longest, count) → interval DP
(longest palindromic subsequence) → cuts on top of centre expansion. Then two strings: LCS
(the core) → deletions = LCS reuse → edit distance (three operations) → interleaving →
counting subsequences → regex and wildcard matching.

## 5.1 Palindromes

Every palindrome has a centre: one character (odd length) or a gap (even length). Expanding
from all 2n − 1 centres visits every palindromic substring in O(n²) total, with O(1)
space.

### 5.1.1 [5. Longest Palindromic Substring](https://leetcode.com/problems/longest-palindromic-substring/) — ★★★ · Medium

**Problem.** Longest palindromic substring.  
**Signal.** Every palindrome has a centre → expand around 2n − 1 centres.  
*New here: Centre expansion.*

**Fine print.** Ask before coding:
- Several of the same length? → Return any.
- Substring, so contiguous? → Yes.

> 💡 **Key insight.** A palindrome grows symmetrically from its centre, so trying every centre and expanding while the ends match finds all of them.

**Brute force →** Check every substring → O(n³). DP table `is_pal[l][r]` → O(n²) time and space. Centre expansion → O(n²) time, O(1) space. (Manacher: O(n).)

**Steps**
1. Loop over 2n − 1 centres; centre `k` starts at `lo = k // 2`, `hi = (k + 1) // 2`.
2. Expand while both ends match.
3. The palindrome is `s[lo + 1 : hi]`; keep the longest.

**Complexity**

- **Time:** O(n²).
- **Space:** O(1).

In [46]:
def longest_palindrome_substring(s: str) -> str:
    """Longest palindromic substring."""
    best = (0, 0)
    # even k: centre on a character; odd k: centre between two
    for centre in range(2 * len(s) - 1):
        lo, hi = centre // 2, (centre + 1) // 2
        # grow while the two ends match
        while lo >= 0 and hi < len(s) and s[lo] == s[hi]:
            lo, hi = lo - 1, hi + 1
        # the last successful palindrome is s[lo + 1 : hi]
        if hi - lo - 1 > best[1] - best[0]:
            best = (lo + 1, hi)
    return s[best[0] : best[1]]


assert longest_palindrome_substring("babad") in ("bab", "aba")
assert longest_palindrome_substring("cbbd") == "bb"
assert longest_palindrome_substring("a") == "a"
print(longest_palindrome_substring("forgeeksskeegfor"))

geeksskeeg


### 5.1.2 [647. Palindromic Substrings](https://leetcode.com/problems/palindromic-substrings/) — ★★★ · Medium

**Problem.** Count palindromic substrings.  
**Signal.** Expand around centres, counting every successful expansion.  
*New here: Counting during expansion instead of keeping the best.*

**Fine print.** Ask before coding:
- Count the same text at different positions separately? → Yes.
- Do single characters count? → Yes.

> 💡 **Key insight.** Each successful expansion step from a centre reveals exactly one new palindrome.

**Brute force →** Check every substring → O(n³). Centre expansion → O(n²).

**Steps**
1. Loop over 2n − 1 centres.
2. While the ends match, count one palindrome and expand.
3. Return the count.

**Complexity**

- **Time:** O(n²).
- **Space:** O(1).

In [47]:
def count_substrings(s: str) -> int:
    """Number of palindromic substrings."""
    count = 0
    for centre in range(2 * len(s) - 1):
        lo, hi = centre // 2, (centre + 1) // 2
        # each successful step is one more palindrome
        while lo >= 0 and hi < len(s) and s[lo] == s[hi]:
            count += 1
            lo, hi = lo - 1, hi + 1
    return count


assert count_substrings("abc") == 3
assert count_substrings("aaa") == 6
print(count_substrings("aaa"))

6


### 5.1.3 [516. Longest Palindromic Subsequence](https://leetcode.com/problems/longest-palindromic-subsequence/) — ★ · Medium

**Problem.** Length of the longest palindromic subsequence.  
**Signal.** Interval DP `dp[l][r]` — or LCS of `s` and `reversed(s)`.  
*New here: Interval DP over `(l, r)`, compressed to one row.*

**Fine print.** Ask before coding:
- Subsequence, not contiguous? → Yes.

> 💡 **Key insight.** If the two ends match, they wrap the best answer inside; otherwise drop one end and keep the better result.

**Brute force →** All subsequences → O(2ⁿ). Interval DP → O(n²).

**Steps**
1. `dp[l][l] = 1`.
2. `s[l] == s[r]` → `dp[l][r] = dp[l + 1][r − 1] + 2`.
3. Otherwise `max(dp[l + 1][r], dp[l][r − 1])`.
4. Fill `l` from right to left (so row `l + 1` is ready), keeping one row and the diagonal.

**Complexity**

- **Time:** O(n²).
- **Space:** O(n).

In [48]:
def longest_palindrome_subseq(s: str) -> int:
    """Longest palindromic subsequence length."""
    n = len(s)
    # dp[r] holds dp[l + 1][r] before being overwritten with dp[l][r]
    dp = [0] * n
    for l in range(n - 1, -1, -1):
        # dp[l + 1][r - 1] for the current r
        prev_diag = 0
        dp[l] = 1
        for r in range(l + 1, n):
            saved = dp[r]
            # matching ends wrap the inner answer
            if s[l] == s[r]:
                dp[r] = prev_diag + 2
            # drop the left end or the right end
            else:
                dp[r] = max(dp[r], dp[r - 1])
            prev_diag = saved
    return dp[-1]


assert longest_palindrome_subseq("bbbab") == 4
assert longest_palindrome_subseq("cbbd") == 2
print(longest_palindrome_subseq("bbbab"))

4


### 5.1.4 [132. Palindrome Partitioning II](https://leetcode.com/problems/palindrome-partitioning-ii/) — ★ · Hard

**Problem.** Minimum cuts so every piece is a palindrome.  
**Signal.** `cuts[i]` = min cuts for `s[:i+1]`, with centre expansion to find palindromes.  
*New here: Combining centre expansion with a prefix DP.*

**Fine print.** Ask before coding:
- Count cuts, not pieces? → Cuts, so the answer is pieces − 1.

> 💡 **Key insight.** Every palindrome `s[lo..hi]` found by expansion offers a way to end the prefix at `hi`: one cut after the best partition of `s[:lo]`.

**Brute force →** Backtracking over partitions → exponential. Palindrome table + prefix DP → O(n²) time and space. Centre expansion + prefix DP → O(n²) time, O(n) space.

**Steps**
1. `cuts[i] = i` (worst case: cut after every character).
2. Expand around every centre.
3. For each palindrome `s[lo..hi]`: `cuts[hi] = min(cuts[hi], 0 if lo == 0 else cuts[lo − 1] + 1)`.
4. Return `cuts[n − 1]`.

**Complexity**

- **Time:** O(n²).
- **Space:** O(n).

In [49]:
def min_cut(s: str) -> int:
    """Fewest cuts into palindromes."""
    n = len(s)
    # worst case: cut between every pair of characters
    cuts = list(range(n))
    for centre in range(2 * n - 1):
        lo, hi = centre // 2, (centre + 1) // 2
        while lo >= 0 and hi < n and s[lo] == s[hi]:
            # s[lo..hi] can be the last piece of s[:hi + 1]
            cuts[hi] = min(cuts[hi], 0 if lo == 0 else cuts[lo - 1] + 1)
            lo, hi = lo - 1, hi + 1
    return cuts[-1]


assert min_cut("aab") == 1
assert min_cut("a") == 0
assert min_cut("ab") == 1
print(min_cut("aab"))

1


## 5.2 Two-string alignment

`dp[i][j]` over prefixes. Write the three moves — match/replace (diagonal), drop from `a`
(up), drop from `b` (left) — and what each costs or contributes. Compress to one row by
saving the diagonal before overwriting it.

### 5.2.1 [1143. Longest Common Subsequence](https://leetcode.com/problems/longest-common-subsequence/) — ★★★ · Medium

**Problem.** Length of the longest common subsequence of two strings.  
**Signal.** The canonical two-string DP.  
*New here: The prefix-pair state `dp[i][j]`.*

**Fine print.** Ask before coding:
- Subsequence? → Yes, not contiguous.
- Return the length, or also the string? → The length; ask about rebuilding the string as a follow-up.

> 💡 **Key insight.** If the last characters match, they end the LCS of the two prefixes; otherwise one of them is unused, so drop it and take the better result.

**Brute force →** All subsequences of one string checked against the other → O(2^m · n). DP → O(mn).

**Steps**
1. `dp[i][j]` = LCS of `a[:i]` and `b[:j]`; row and column 0 are 0.
2. Match → `dp[i − 1][j − 1] + 1`.
3. No match → `max(dp[i − 1][j], dp[i][j − 1])`.
4. Keep one row and the saved diagonal.

**Complexity**

- **Time:** O(mn).
- **Space:** O(n).

In [50]:
def longest_common_subsequence(a: str, b: str) -> int:
    """LCS length."""
    # dp[j] = LCS of the current prefix of a and b[:j]
    dp = [0] * (len(b) + 1)
    for ch in a:
        # dp[i - 1][j - 1]
        diag = 0
        for j in range(1, len(b) + 1):
            above = dp[j]
            # match extends the diagonal, otherwise drop one character
            dp[j] = diag + 1 if ch == b[j - 1] else max(dp[j], dp[j - 1])
            diag = above
    return dp[-1]


assert longest_common_subsequence("abcde", "ace") == 3
assert longest_common_subsequence("abc", "def") == 0
print(longest_common_subsequence("abcde", "ace"))

3


### 5.2.2 [583. Delete Operation for Two Strings](https://leetcode.com/problems/delete-operation-for-two-strings/) — ★ · Medium

**Problem.** Fewest deletions (from either string) to make them equal.  
**Signal.** Keep the LCS, delete everything else.  
*New here: Reusing LCS through a formula.*

**Fine print.** Ask before coding:
- Deletions only? → Yes, so the answer is `m + n − 2·LCS`.

> 💡 **Key insight.** The longest string you can keep in both is their LCS; everything else must be deleted.

**Brute force →** A separate deletion DP → same O(mn). The LCS formula is one line.

**Steps**
1. Compute `LCS(a, b)`.
2. Return `len(a) + len(b) − 2 · LCS`.

**Complexity**

- **Time:** O(mn).
- **Space:** O(n).

In [51]:
def min_distance_delete(a: str, b: str) -> int:
    """Deletions to make two strings equal."""
    # keep the LCS in both strings, delete everything else
    return len(a) + len(b) - 2 * longest_common_subsequence(a, b)


assert min_distance_delete("sea", "eat") == 2
assert min_distance_delete("leetcode", "etco") == 4
print(min_distance_delete("sea", "eat"))

2


### 5.2.3 [72. Edit Distance](https://leetcode.com/problems/edit-distance/) — ★★ · Medium

**Problem.** Fewest inserts, deletes and replaces to turn `a` into `b`.  
**Signal.** Two-string DP with three operations.  
*New here: Three transitions with costs, and non-zero base cases.*

**Fine print.** Ask before coding:
- Does each operation cost 1? → Yes.
- Can either string be empty? → Yes.

> 💡 **Key insight.** Look at the last characters: equal → free; otherwise the last edit is a replace (diagonal), a delete from `a` (up) or an insert into `a` (left), each costing 1.

**Brute force →** Try all edit sequences → exponential. DP → O(mn).

**Steps**
1. `dp[i][0] = i` (delete everything), `dp[0][j] = j` (insert everything).
2. Equal last characters → `dp[i − 1][j − 1]`.
3. Otherwise `1 + min(replace, delete, insert)`.
4. One row + saved diagonal; `dp[0]` becomes `i` at each new row.

**Complexity**

- **Time:** O(mn).
- **Space:** O(n).

In [52]:
def min_distance(a: str, b: str) -> int:
    """Levenshtein distance."""
    # row 0: turning "" into b[:j] needs j inserts
    dp = list(range(len(b) + 1))
    for i, ch in enumerate(a, 1):
        # diag = dp[i - 1][0]; turning a[:i] into "" needs i deletes
        diag, dp[0] = dp[0], i
        for j in range(1, len(b) + 1):
            above = dp[j]
            # last characters equal: no edit needed
            if ch == b[j - 1]:
                dp[j] = diag
            # replace (diag), delete (above) or insert (left)
            else:
                dp[j] = 1 + min(diag, above, dp[j - 1])
            diag = above
    return dp[-1]


assert min_distance("horse", "ros") == 3
assert min_distance("intention", "execution") == 5
assert min_distance("", "abc") == 3
print(min_distance("horse", "ros"))

3


### 5.2.4 [97. Interleaving String](https://leetcode.com/problems/interleaving-string/) — ★★ · Medium

**Problem.** Is `s3` an interleaving of `s1` and `s2` (keeping each one's order)?  
**Signal.** `dp[i][j]` = `s3[:i+j]` is an interleaving of `s1[:i]` and `s2[:j]`.  
*New here: A third string indexed by `i + j`.*

**Fine print.** Ask before coding:
- Must `len(s3) == len(s1) + len(s2)`? → Check it first.
- O(n) space? → That is the follow-up.

> 💡 **Key insight.** The last character of `s3[:i + j]` came from either `s1[i − 1]` or `s2[j − 1]`, and the rest must be a valid interleaving of the shorter prefixes.

**Brute force →** Recursion choosing the source of each character → O(2^(m+n)). DP → O(mn).

**Steps**
1. Lengths must add up.
2. First row: prefixes of `s2` alone must match `s3`.
3. Each new row: column 0 uses `s1` alone.
4. `dp[j] = (dp[j] and s1[i−1] == s3[i+j−1]) or (dp[j−1] and s2[j−1] == s3[i+j−1])`.

**Complexity**

- **Time:** O(mn).
- **Space:** O(n).

In [53]:
def is_interleave(s1: str, s2: str, s3: str) -> bool:
    """True if s3 interleaves s1 and s2."""
    if len(s1) + len(s2) != len(s3):
        return False
    # row 0: only characters from s2
    dp = [True] + [False] * len(s2)
    for j in range(1, len(s2) + 1):
        dp[j] = dp[j - 1] and s2[j - 1] == s3[j - 1]
    for i in range(1, len(s1) + 1):
        # column 0: only characters from s1
        dp[0] = dp[0] and s1[i - 1] == s3[i - 1]
        for j in range(1, len(s2) + 1):
            # last character came from s1 (above) or from s2 (left)
            dp[j] = (dp[j] and s1[i - 1] == s3[i + j - 1]) or (
                dp[j - 1] and s2[j - 1] == s3[i + j - 1]
            )
    return dp[-1]


assert is_interleave("aabcc", "dbbca", "aadbbcbcac")
assert not is_interleave("aabcc", "dbbca", "aadbbbaccc")
assert is_interleave("", "", "")
print(is_interleave("aabcc", "dbbca", "aadbbcbcac"))

True


### 5.2.5 [115. Distinct Subsequences](https://leetcode.com/problems/distinct-subsequences/) — ★★ · Hard

**Problem.** Number of distinct subsequences of `s` equal to `t`.  
**Signal.** Counting two-string DP.  
*New here: Counting with "skip" and "use" transitions, one row iterated backwards.*

**Fine print.** Ask before coding:
- Count by positions used? → Yes, distinct sets of indices.
- Does the answer fit in 32 bits? → Yes, guaranteed.

> 💡 **Key insight.** Each character of `s` can be skipped (keep the old count) or, if it matches `t[j − 1]`, used to finish `t[:j]` (add the count for `t[:j − 1]`).

**Brute force →** Enumerate subsequences of `s` → O(2ⁿ). DP → O(mn).

**Steps**
1. `dp[0] = 1` (the empty `t` appears once), others 0.
2. For each character of `s`, for `j` from `len(t)` down to 1:
3. if it matches `t[j − 1]`, `dp[j] += dp[j − 1]`.
4. Return `dp[len(t)]`.

**Complexity**

- **Time:** O(mn).
- **Space:** O(n).

⚠️ **Trap.** Iterating `j` upwards would let one character of `s` be used twice.

In [54]:
def num_distinct(s: str, t: str) -> int:
    """Ways to form t as a subsequence of s."""
    # dp[j] = ways the prefix of s seen so far forms t[:j]
    dp = [1] + [0] * len(t)
    for ch in s:
        # backwards so dp[j - 1] is still the value before this character
        for j in range(len(t), 0, -1):
            if ch == t[j - 1]:
                dp[j] += dp[j - 1]
    return dp[-1]


assert num_distinct("rabbbit", "rabbit") == 3
assert num_distinct("babgbag", "bag") == 5
print(num_distinct("babgbag", "bag"))

5


### 5.2.6 [10. Regular Expression Matching](https://leetcode.com/problems/regular-expression-matching/) — ★★ · Hard

**Problem.** Match `s` against pattern `p` with `.` (any char) and `*` (zero or more of the previous).  
**Signal.** Two-string DP; `*` either uses zero copies or consumes one character of `s`.  
*New here: Top-down two-string DP for matching, with a two-way choice on `*`.*

**Fine print.** Ask before coding:
- Must the whole string match, not a prefix? → The whole string.
- Is `*` always preceded by a valid character? → Yes.

> 💡 **Key insight.** `x*` either matches nothing (skip it) or matches one character and stays available for more — two smaller subproblems.

**Brute force →** Plain recursion → exponential (repeated `(i, j)` pairs). Memoised → O(mn).

**Steps**
1. `match(i, j)` = does `s[i:]` match `p[j:]`; end of pattern → end of string?
2. `first` = current characters match (`.` matches anything).
3. Next pattern char is `*` → `match(i, j + 2)` or (`first` and `match(i + 1, j)`).
4. Otherwise `first and match(i + 1, j + 1)`.

**Complexity**

- **Time:** O(mn).
- **Space:** O(mn) — the cache.

In [55]:
def is_match_regex(s: str, p: str) -> bool:
    """Regex match supporting . and *."""

    @cache
    def match(i: int, j: int) -> bool:
        """Does s[i:] match p[j:]?"""
        if j == len(p):
            return i == len(s)
        # does the current character match p[j]?
        first = i < len(s) and p[j] in (s[i], ".")
        if j + 1 < len(p) and p[j + 1] == "*":
            # use zero copies, or consume one char and keep the star
            return match(i, j + 2) or (first and match(i + 1, j))
        return first and match(i + 1, j + 1)

    return match(0, 0)


assert not is_match_regex("aa", "a")
assert is_match_regex("aa", "a*")
assert is_match_regex("ab", ".*")
assert is_match_regex("aab", "c*a*b")
assert not is_match_regex("mississippi", "mis*is*p*.")
print(is_match_regex("aab", "c*a*b"))

True


### 5.2.7 [44. Wildcard Matching](https://leetcode.com/problems/wildcard-matching/) — ★ · Hard

**Problem.** Match with `?` (any single char) and `*` (any sequence, including empty).  
**Signal.** Two-string DP; `*` either matches empty or eats one more character.  
*New here: A standalone `*` (no preceding character), bottom-up with one row.*

**Fine print.** Ask before coding:
- Must the whole string match? → Yes.
- Can `*` match an empty sequence? → Yes.

> 💡 **Key insight.** `*` matches empty (`dp[i][j − 1]`) or swallows one more character of `s` while staying active (`dp[i − 1][j]`).

**Brute force →** Recursion over every split for each `*` → exponential. DP → O(mn).

**Steps**
1. Row 0: `dp[0][j]` is true only for a prefix of `*`s.
2. For each character of `s`: column 0 becomes false.
3. `*` → `dp[i − 1][j] or dp[i][j − 1]`; `?` or equal → `dp[i − 1][j − 1]`.
4. Return the last cell.

**Complexity**

- **Time:** O(mn).
- **Space:** O(n).

In [56]:
def is_match_wildcard(s: str, p: str) -> bool:
    """Wildcard match supporting ? and *."""
    # row 0: the empty string matches only leading stars
    dp = [True] + [False] * len(p)
    for j in range(1, len(p) + 1):
        dp[j] = dp[j - 1] and p[j - 1] == "*"
    for ch in s:
        # non-empty s never matches an empty pattern
        diag, dp[0] = dp[0], False
        for j in range(1, len(p) + 1):
            above = dp[j]
            # star: swallow this char (above) or match empty (left)
            if p[j - 1] == "*":
                dp[j] = dp[j] or dp[j - 1]
            # single-character match uses the diagonal
            else:
                dp[j] = diag and p[j - 1] in (ch, "?")
            diag = above
    return dp[-1]


assert not is_match_wildcard("aa", "a")
assert is_match_wildcard("aa", "*")
assert not is_match_wildcard("cb", "?a")
assert is_match_wildcard("adceb", "*a*b")
print(is_match_wildcard("adceb", "*a*b"))

True


---
# 6) Interval and Game DP

**In plain English.** Interval DP solves a problem on a range `[l, r]` by trying every way
to split it and combining answers for the smaller ranges. Game DP is the same with two
players: your score minus your opponent's best score on what remains.

> **Analogy.** Planning demolition of a row of buildings: decide which building comes down
> *last* — then the left and right blocks are independent jobs you already know how to
> price.

`dp[l][r]` describes the subarray `l..r`. Fill by increasing interval length (or memoise),
so smaller intervals are ready. Choosing the element handled **last** in the interval (not
first) makes the two sides independent.

**Recognition signals**

- "burst", "merge stones", "cut a stick", "remove boxes" — cost depends on neighbours
- two players taking from the ends of an array, "both play optimally"
- n ≤ 500 hints at O(n³)
- weighted interval scheduling: "non-overlapping jobs with profit" → sort + binary search

**Common bugs**

- Choosing the *first* action in the interval → the sides still interact.
- Filling by `l` then `r` instead of by length, reading uncomputed states.
- Game DP that tracks both players' totals separately instead of the score difference.

**Don't confuse with**

| Pattern | Difference |
|---|---|
| Two-string DP | `dp[i][j]` over two prefixes; interval DP is one sequence's sub-range |
| Greedy interval scheduling | maximises the *count* of intervals; weighted needs DP |
| Divide and conquer | the split point is fixed; interval DP tries every split |

**Learning path.** Last-action trick (Burst Balloons) → same shape with cuts → minimax as a
difference (Predict the Winner) → a game you can win by argument (Stone Game) → divisor
DP that reduces to prime factors (2 Keys) → DP + binary search (Job Scheduling).

## 6.1 Interval DP

Write `dp(l, r)` as "best result for the range `l..r`", then loop over the split or last
action `k` inside it. Use `@cache` on a recursive version first; convert to a table only
if needed.

### 6.1.1 [312. Burst Balloons](https://leetcode.com/problems/burst-balloons/) — ★★ · Hard

**Problem.** Bursting balloon `i` earns `left × nums[i] × right` with its current neighbours. Maximise coins.  
**Signal.** Choose the balloon burst **last** in each interval — its neighbours are then fixed.  
*New here: The "last action in the interval" trick.*

**Fine print.** Ask before coding:
- Are virtual balloons with value 1 at both ends? → Yes, so pad with 1 on each side.
- `n ≤ 300`? → Yes, so O(n³) is fine.

> 💡 **Key insight.** If `k` is the last balloon burst between `l` and `r`, its neighbours at that moment are exactly `l` and `r`, and the two sides were burst independently.

**Brute force →** Try every burst order → O(n!). Interval DP → O(n³).

**Steps**
1. Pad with 1 on both ends.
2. `dp[l][r]` = best coins from bursting everything strictly between `l` and `r`.
3. For each length, each `l`, try every last balloon `k`: `dp[l][k] + vals[l]·vals[k]·vals[r] + dp[k][r]`.
4. Return `dp[0][n − 1]`.

**Complexity**

- **Time:** O(n³).
- **Space:** O(n²).

In [57]:
def max_coins(nums: list[int]) -> int:
    """Max coins from bursting all balloons."""
    # virtual balloons of value 1 at both ends
    vals = [1] + nums + [1]
    n = len(vals)
    # dp[l][r] = best for the balloons strictly between l and r
    dp = [[0] * n for _ in range(n)]
    # shorter intervals first
    for length in range(2, n):
        for l in range(n - length):
            r = l + length
            # k is burst last, so its neighbours are l and r
            dp[l][r] = max(
                dp[l][k] + vals[l] * vals[k] * vals[r] + dp[k][r]
                for k in range(l + 1, r)
            )
    return dp[0][n - 1]


assert max_coins([3, 1, 5, 8]) == 167
assert max_coins([1, 5]) == 10
print(max_coins([3, 1, 5, 8]))

167


### 6.1.2 [1547. Minimum Cost to Cut a Stick](https://leetcode.com/problems/minimum-cost-to-cut-a-stick/) — ★ · Hard

**Problem.** Cut a stick of length `n` at given positions; each cut costs the current piece's length. Minimise total cost.  
**Signal.** Burst Balloons in disguise: the first cut in an interval splits it into two independent intervals.  
*New here: Interval DP over *positions* (cut points) instead of array indices.*

**Fine print.** Ask before coding:
- Can cuts happen in any order? → Yes, that is the choice to optimise.
- Are the cut positions unique? → Yes.

> 💡 **Key insight.** The first cut inside a piece costs the piece's length and splits it into two independent pieces.

**Brute force →** Try every cut order → O(m!). Interval DP over cut points → O(m³).

**Steps**
1. Positions = `[0] + sorted(cuts) + [n]`.
2. `cost(i, j)` = cheapest way to make every cut strictly between positions `i` and `j`.
3. No cut inside → 0.
4. Otherwise `pos[j] − pos[i] + min(cost(i, k) + cost(k, j))`.

**Complexity**

- **Time:** O(m³).
- **Space:** O(m²).

In [58]:
def min_cost_cut(n: int, cuts: list[int]) -> int:
    """Cheapest order of cutting the stick."""
    pos = [0] + sorted(cuts) + [n]

    @cache
    def cost(i: int, j: int) -> int:
        """Cheapest way to make all cuts between pos[i] and pos[j]."""
        # no cut point strictly inside
        if j - i < 2:
            return 0
        # first cut at k costs this piece's length
        return pos[j] - pos[i] + min(cost(i, k) + cost(k, j) for k in range(i + 1, j))

    return cost(0, len(pos) - 1)


assert min_cost_cut(7, [1, 3, 4, 5]) == 16
assert min_cost_cut(9, [5, 6, 1, 4, 2]) == 22
print(min_cost_cut(7, [1, 3, 4, 5]))

16


### 6.1.3 [486. Predict the Winner](https://leetcode.com/problems/predict-the-winner/) — ★ · Medium

**Problem.** Players alternately take from either end of the array; can player 1 win (ties count)?  
**Signal.** Minimax as a score *difference*: `diff(l, r)` = current player's best margin.  
*New here: Game DP with the score-difference trick.*

**Fine print.** Ask before coding:
- Does a tie count as a win for player 1? → Yes.
- Do both players play optimally? → Yes.

> 💡 **Key insight.** After you take a number, the opponent faces the same game on the rest, so your margin is what you took minus their best margin.

**Brute force →** Explore the full game tree → O(2ⁿ). Memoised margin → O(n²).

**Steps**
1. `diff(l, l) = nums[l]`.
2. `diff(l, r) = max(nums[l] − diff(l + 1, r), nums[r] − diff(l, r − 1))`.
3. Player 1 wins iff `diff(0, n − 1) ≥ 0`.

**Complexity**

- **Time:** O(n²).
- **Space:** O(n²).

In [59]:
def predict_the_winner(nums: list[int]) -> bool:
    """True if player 1 can at least tie."""

    @cache
    def diff(l: int, r: int) -> int:
        """Best (my score - opponent score) on nums[l..r] for the player to move."""
        if l == r:
            return nums[l]
        # take an end, then the opponent plays optimally on the rest
        return max(nums[l] - diff(l + 1, r), nums[r] - diff(l, r - 1))

    return diff(0, len(nums) - 1) >= 0


assert not predict_the_winner([1, 5, 2])
assert predict_the_winner([1, 5, 233, 7])
print(predict_the_winner([1, 5, 233, 7]))

True


### 6.1.4 [877. Stone Game](https://leetcode.com/problems/stone-game/) — ★ · Medium

**Problem.** Even number of piles, odd total; players take from either end. Does Alice win?  
**Signal.** Same margin DP — and a parity argument says the first player always wins.  
*New here: A proof that replaces the DP.*

**Fine print.** Ask before coding:
- Even number of piles and an odd total? → Yes, which is why Alice always wins.

> 💡 **Key insight.** With an even number of piles, Alice can force taking all even-indexed piles or all odd-indexed piles; one of those sets is strictly larger (the total is odd), so she always wins.

**Brute force →** The margin DP → O(n²). The argument → O(1) (`return True`).

**Steps**
1. Reuse the margin DP to confirm (or just return `True`).

**Complexity**

- **Time:** O(n²) with the DP, O(1) by the argument.
- **Space:** O(n²) with the DP.

In [60]:
def stone_game(piles: list[int]) -> bool:
    """True if Alice wins (always, for even pile counts)."""
    # the parity argument says True; the DP confirms it
    return predict_the_winner(piles)


assert stone_game([5, 3, 4, 5])
assert stone_game([3, 7, 2, 3])
print(stone_game([5, 3, 4, 5]))

True


### 6.1.5 [650. 2 Keys Keyboard](https://leetcode.com/problems/2-keys-keyboard/) — ★ · Medium

**Problem.** Start with one `A`; operations Copy All and Paste. Fewest operations to get `n` `A`s?  
**Signal.** Building `n` from a divisor `d` costs `n / d` operations (1 copy + `n/d − 1` pastes) → sum of prime factors.  
*New here: A DP whose closed form is the sum of prime factors.*

**Fine print.** Ask before coding:
- Does Copy All copy everything currently on screen? → Yes.
- `n = 1`? → 0 operations.

> 💡 **Key insight.** Going from `d` characters to `n` (a multiple of `d`) costs `n / d` operations, and splitting a factor `a·b` into two steps never costs more (`a + b ≤ a·b`), so the answer is the sum of prime factors.

**Brute force →** BFS over (characters, clipboard) states → O(n²). Divisor DP → O(n √n). Factorisation → O(√n).

**Steps**
1. Try divisors `d = 2, 3, …`.
2. While `d` divides `n`, add `d` and divide.
3. Return the total.

**Complexity**

- **Time:** O(√n) (trial division; here it runs up to the largest prime factor).
- **Space:** O(1).

In [61]:
def min_steps(n: int) -> int:
    """Fewest copy/paste operations to reach n characters."""
    total, d = 0, 2
    while n > 1:
        # each prime factor d costs one copy + (d - 1) pastes
        while n % d == 0:
            total += d
            n //= d
        d += 1
    return total


assert min_steps(3) == 3
assert min_steps(1) == 0
assert min_steps(12) == 7
print(min_steps(12))

7


## 6.2 DP with binary search

Sort the items so "compatible earlier items" form a prefix, then use binary search to find
that prefix inside the DP transition.

### 6.2.1 [1235. Maximum Profit in Job Scheduling](https://leetcode.com/problems/maximum-profit-in-job-scheduling/) — ★★ · Hard

**Problem.** Jobs `(start, end, profit)`; choose non-overlapping jobs maximising profit.  
**Signal.** Weighted interval scheduling: sort by end, DP + binary search for the last compatible job.  
*New here: DP with a binary-searched predecessor.*

**Fine print.** Ask before coding:
- Can a job start exactly when another ends? → Yes, that is not an overlap.
- Are the jobs unsorted? → Yes, sort by end time.

> 💡 **Key insight.** After sorting by end time, the jobs compatible with job `i` are exactly those ending by its start — a prefix you can find with `bisect_right`.

**Brute force →** Try every subset → O(2ⁿ). DP with a linear scan for the predecessor → O(n²). With binary search → O(n log n).

**Steps**
1. Sort jobs by end time.
2. `dp[i]` = best profit using the first `i` jobs.
3. `k` = number of jobs ending at or before job `i`'s start (bisect on end times).
4. `dp[i] = max(dp[i − 1], profit + dp[k])`.

**Complexity**

- **Time:** O(n log n).
- **Space:** O(n).

In [62]:
def job_scheduling(start: list[int], end: list[int], profit: list[int]) -> int:
    """Max profit from non-overlapping jobs."""
    jobs = sorted(zip(end, start, profit))
    ends = [e for e, _, _ in jobs]
    # dp[i] = best profit using the first i jobs (by end time)
    dp = [0] * (len(jobs) + 1)
    for i, (_, s, p) in enumerate(jobs, 1):
        # earlier jobs that end by this job's start
        k = bisect_right(ends, s, hi=i - 1)
        # skip this job, or take it after the compatible prefix
        dp[i] = max(dp[i - 1], p + dp[k])
    return dp[-1]


assert job_scheduling([1, 2, 3, 3], [3, 4, 5, 6], [50, 10, 40, 70]) == 120
assert job_scheduling([1, 2, 3, 4, 6], [3, 5, 10, 6, 9], [20, 20, 100, 70, 60]) == 150
assert job_scheduling([1, 1, 1], [2, 3, 4], [5, 6, 4]) == 6
print(job_scheduling([1, 2, 3, 3], [3, 4, 5, 6], [50, 10, 40, 70]))

120


---
# Summary

| Family | Recognise it by | State → transition | Cost |
|---|---|---|---|
| Fibonacci-style | ways / best up to i, adjacency ban | `dp[i]` from `dp[i−1]`, `dp[i−2]` | O(n), O(1) |
| Kadane | best contiguous subarray (sum, product, circular) | best ending at i: extend or restart | O(n) |
| Word / partition | segment a string, split a number | `dp[i]` over prefix lengths | O(n·L) |
| LIS | increasing / chained subsequence | patience `tails` + binary search | O(n log n) |
| 0/1 knapsack | subset sum, partition, target sum | capacity loop **downwards** | O(n·W) |
| Unbounded knapsack | coins, squares | capacity loop **upwards**; loop order = combos vs sequences | O(n·W) |
| State machine | stock buy/sell with limits | one variable per state, yesterday's values | O(n·k) |
| Grid | right/down paths | one row array, in-place update | O(mn), O(n) |
| Backward grid | requirement depends on the future | fill from the end | O(mn) |
| Memo DFS on DAG | longest increasing path | `@cache` on cell | O(mn) |
| Palindromes | substrings / subsequences / cuts | expand around centres; `dp[l][r]` | O(n²) |
| Two strings | LCS, edit distance, interleave, regex | `dp[i][j]` over prefixes | O(mn) |
| Interval | burst balloons, cut stick, games | choose the **last** action in `[l, r]` | O(n³) |
| Weighted intervals | job scheduling | sort by end + bisect | O(n log n) |